In [ ]:
# ============================================================
# REPRODUCIBILITY PATH SETUP
# ============================================================

from pathlib import Path
import os

# Change TRAFFIC_FORECASTING_ROOT only if the project is stored
# somewhere other than the default Colab location.
PROJECT_ROOT = Path(
    os.environ.get(
        "TRAFFIC_FORECASTING_ROOT",
        "/content/traffic-forecasting-failure-robustness"
    )
)

RAW_DATA_ROOT = PROJECT_ROOT / "data" / "raw"
OUTPUT_ROOT = PROJECT_ROOT / "outputs"

RAW_DATA_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Project root :", PROJECT_ROOT)
print("Raw data     :", RAW_DATA_ROOT)
print("Outputs      :", OUTPUT_ROOT)


> **Reproducibility note.** This public version uses project-relative paths. By default, the project root is `/content/traffic-forecasting-failure-robustness`. To use another location (including Google Drive), set the environment variable `TRAFFIC_FORECASTING_ROOT` before running the notebook. Raw source archives belong in `data/raw/`; generated artifacts are written under `outputs/`.


In [ ]:
from pathlib import Path

# Raw monthly ZIP archives downloaded from the official DataVic source
DATA_FOLDER = RAW_DATA_ROOT

zip_files = sorted(DATA_FOLDER.glob("*.zip"))

print("===== ZIP FILE CHECK =====")
print(f"ZIP files found: {len(zip_files)}")

if zip_files:
    print("\nFiles found:")
    for f in zip_files:
        print(f.name)
else:
    print("❌ No ZIP files found. Check DATA_FOLDER path.")

In [ ]:
import zipfile
import re
from datetime import datetime

print("===== ZIP CONTENT CHECK =====")

total_csv = 0
all_dates = []

for zip_path in zip_files:
    with zipfile.ZipFile(zip_path, "r") as z:

        csv_names = [
            name for name in z.namelist()
            if name.lower().endswith(".csv")
        ]

        total_csv += len(csv_names)

        # Extract dates from names like VSDATA_20260101.csv
        dates = []

        for name in csv_names:
            match = re.search(r"VSDATA_(\d{8})\.csv", name, re.IGNORECASE)

            if match:
                date = datetime.strptime(
                    match.group(1), "%Y%m%d"
                ).date()

                dates.append(date)
                all_dates.append(date)

        print(f"\n{zip_path.name}")
        print(f"CSV files: {len(csv_names)}")

        if dates:
            print(f"Date range: {min(dates)} → {max(dates)}")


print("\n===== OVERALL CHECK =====")
print(f"Total CSV files: {total_csv}")

if all_dates:
    print(f"Overall period: {min(all_dates)} → {max(all_dates)}")
    print(f"Unique dates: {len(set(all_dates))}")

In [ ]:
import pandas as pd
import numpy as np
import zipfile
import gc
from pathlib import Path

# ============================================================
# NOTEBOOK 2 — BUILD SITE-DETECTOR SUMMARY
# ============================================================

# Columns needed only
USE_COLS = [
    "NB_SCATS_SITE",
    "NB_DETECTOR",
    "QT_VOLUME_24HOUR",
    "CT_RECORDS",
    "CT_ALARM_24HOUR"
]

# Output folder
OUTPUT_FOLDER = OUTPUT_ROOT / "Notebook_2_Output"
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

overall_summary = None

print("===== BUILDING SITE-DETECTOR SUMMARY =====\n")

for zip_path in zip_files:

    print(f"Processing: {zip_path.name}")

    daily_frames = []

    with zipfile.ZipFile(zip_path, "r") as z:

        csv_names = sorted([
            name for name in z.namelist()
            if name.lower().endswith(".csv")
        ])

        for csv_name in csv_names:

            with z.open(csv_name) as f:

                df = pd.read_csv(
                    f,
                    usecols=USE_COLS,
                    dtype={
                        "NB_SCATS_SITE": "int32",
                        "NB_DETECTOR": "int16",
                        "QT_VOLUME_24HOUR": "int32",
                        "CT_RECORDS": "int16",
                        "CT_ALARM_24HOUR": "int16"
                    }
                )

            # ------------------------------------------------
            # Create daily quality indicators
            # ------------------------------------------------

            df["Days_Present"] = 1

            df["Complete_Days"] = (
                df["CT_RECORDS"] == 96
            ).astype("int8")

            df["Incomplete_Days"] = (
                df["CT_RECORDS"] < 96
            ).astype("int8")

            df["Alarm_Days"] = (
                df["CT_ALARM_24HOUR"] > 0
            ).astype("int8")

            df["Positive_Volume_Days"] = (
                df["QT_VOLUME_24HOUR"] > 0
            ).astype("int8")

            df["Zero_Volume_Days"] = (
                df["QT_VOLUME_24HOUR"] == 0
            ).astype("int8")

            df["Negative_Volume_Days"] = (
                df["QT_VOLUME_24HOUR"] < 0
            ).astype("int8")

            # Volume only when daily volume is positive
            df["Positive_Volume_Sum"] = np.where(
                df["QT_VOLUME_24HOUR"] > 0,
                df["QT_VOLUME_24HOUR"],
                0
            )

            daily_frames.append(
                df[
                    [
                        "NB_SCATS_SITE",
                        "NB_DETECTOR",
                        "Days_Present",
                        "Complete_Days",
                        "Incomplete_Days",
                        "Alarm_Days",
                        "Positive_Volume_Days",
                        "Zero_Volume_Days",
                        "Negative_Volume_Days",
                        "Positive_Volume_Sum"
                    ]
                ]
            )

    # Combine only one month
    month_df = pd.concat(
        daily_frames,
        ignore_index=True
    )

    month_rows = len(month_df)

    # Aggregate month to Site-Detector level
    month_summary = (
        month_df
        .groupby(
            ["NB_SCATS_SITE", "NB_DETECTOR"],
            sort=False
        )
        .sum()
    )

    # Add month summary to overall summary
    if overall_summary is None:
        overall_summary = month_summary
    else:
        overall_summary = overall_summary.add(
            month_summary,
            fill_value=0
        )

    print(f"  Rows processed: {month_rows:,}")
    print(
        f"  Site-Detector pairs: "
        f"{len(month_summary):,}"
    )

    # Free memory
    del daily_frames
    del month_df
    del month_summary
    gc.collect()

    print("  ✅ Done\n")


# ============================================================
# FINALIZE SUMMARY
# ============================================================

overall_summary = overall_summary.reset_index()

count_columns = [
    "Days_Present",
    "Complete_Days",
    "Incomplete_Days",
    "Alarm_Days",
    "Positive_Volume_Days",
    "Zero_Volume_Days",
    "Negative_Volume_Days"
]

overall_summary[count_columns] = (
    overall_summary[count_columns]
    .fillna(0)
    .astype("int32")
)

overall_summary["Positive_Volume_Sum"] = (
    overall_summary["Positive_Volume_Sum"]
    .fillna(0)
    .astype("int64")
)

# ------------------------------------------------------------
# Derived quality indicators
# ------------------------------------------------------------

TOTAL_DAYS = 242

overall_summary["Presence_%"] = (
    overall_summary["Days_Present"]
    / TOTAL_DAYS * 100
)

overall_summary["Complete_%"] = (
    overall_summary["Complete_Days"]
    / overall_summary["Days_Present"] * 100
)

overall_summary["Alarm_%"] = (
    overall_summary["Alarm_Days"]
    / overall_summary["Days_Present"] * 100
)

overall_summary["Positive_Volume_%"] = (
    overall_summary["Positive_Volume_Days"]
    / overall_summary["Days_Present"] * 100
)

overall_summary["Mean_Positive_Daily_Volume"] = np.where(
    overall_summary["Positive_Volume_Days"] > 0,
    overall_summary["Positive_Volume_Sum"]
    / overall_summary["Positive_Volume_Days"],
    0
)

# ============================================================
# SAVE
# ============================================================

output_file = (
    OUTPUT_FOLDER /
    "site_detector_summary_raw.csv"
)

overall_summary.to_csv(
    output_file,
    index=False
)

print("========================================")
print("SITE-DETECTOR SUMMARY COMPLETE")
print("========================================")

print(
    f"Total Site-Detector pairs: "
    f"{len(overall_summary):,}"
)

print(
    f"Sites: "
    f"{overall_summary['NB_SCATS_SITE'].nunique():,}"
)

print(
    f"Maximum days present: "
    f"{overall_summary['Days_Present'].max()}"
)

print(
    f"Pairs present all 242 days: "
    f"{(overall_summary['Days_Present'] == 242).sum():,}"
)

print("\nSaved to:")
print(output_file)

In [ ]:
# ============================================================
# NOTEBOOK 2 — SITE-DETECTOR QUALITY DISTRIBUTIONS
# ============================================================

import pandas as pd
import numpy as np

summary = pd.read_csv(
    OUTPUT_FOLDER / "site_detector_summary_raw.csv"
)

print("========================================")
print("SITE-DETECTOR QUALITY DISTRIBUTIONS")
print("========================================")

print(f"\nTotal Site-Detector pairs: {len(summary):,}")

# ------------------------------------------------------------
# 1. DAYS PRESENT
# ------------------------------------------------------------

print("\n===== DAYS PRESENT =====")

print(
    summary["Days_Present"]
    .describe(
        percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

print("\nPresence counts:")
for days in [242, 240, 230, 220, 200, 180, 120, 60, 30]:
    count = (summary["Days_Present"] >= days).sum()

    print(
        f">= {days:3d} days: "
        f"{count:,} "
        f"({count / len(summary) * 100:.2f}%)"
    )


# ------------------------------------------------------------
# 2. COMPLETENESS
# ------------------------------------------------------------

print("\n===== COMPLETE DAYS % =====")

print(
    summary["Complete_%"]
    .describe(
        percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

print("\nCompleteness thresholds:")
for threshold in [100, 99.5, 99, 98, 95, 90]:
    count = (
        summary["Complete_%"] >= threshold
    ).sum()

    print(
        f">= {threshold:5.1f}% complete: "
        f"{count:,} "
        f"({count / len(summary) * 100:.2f}%)"
    )


# ------------------------------------------------------------
# 3. ALARM DAYS
# ------------------------------------------------------------

print("\n===== ALARM DAYS % =====")

print(
    summary["Alarm_%"]
    .describe(
        percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

print("\nAlarm thresholds:")
for threshold in [0, 0.5, 1, 2, 5, 10, 20]:
    count = (
        summary["Alarm_%"] <= threshold
    ).sum()

    print(
        f"<= {threshold:4.1f}% alarm days: "
        f"{count:,} "
        f"({count / len(summary) * 100:.2f}%)"
    )


# ------------------------------------------------------------
# 4. POSITIVE VOLUME DAYS
# ------------------------------------------------------------

print("\n===== POSITIVE VOLUME DAYS % =====")

print(
    summary["Positive_Volume_%"]
    .describe(
        percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

print("\nPositive-volume thresholds:")
for threshold in [100, 99, 98, 95, 90, 80, 50]:
    count = (
        summary["Positive_Volume_%"] >= threshold
    ).sum()

    print(
        f">= {threshold:3d}% positive days: "
        f"{count:,} "
        f"({count / len(summary) * 100:.2f}%)"
    )


# ------------------------------------------------------------
# 5. ZERO / NEGATIVE DAILY VOLUME
# ------------------------------------------------------------

print("\n===== ZERO / NEGATIVE VOLUME =====")

zero_any = (
    summary["Zero_Volume_Days"] > 0
).sum()

negative_any = (
    summary["Negative_Volume_Days"] > 0
).sum()

no_zero_negative = (
    (summary["Zero_Volume_Days"] == 0) &
    (summary["Negative_Volume_Days"] == 0)
).sum()

print(
    f"Pairs with ≥1 zero-volume day: "
    f"{zero_any:,}"
)

print(
    f"Pairs with ≥1 negative-volume day: "
    f"{negative_any:,}"
)

print(
    f"Pairs with NO zero/negative days: "
    f"{no_zero_negative:,}"
)


# ------------------------------------------------------------
# 6. TRAFFIC VOLUME
# ------------------------------------------------------------

print("\n===== MEAN POSITIVE DAILY VOLUME =====")

print(
    summary["Mean_Positive_Daily_Volume"]
    .describe(
        percentiles=[
            0.01, 0.05, 0.10, 0.25,
            0.50, 0.75, 0.90, 0.95, 0.99
        ]
    )
)


# ------------------------------------------------------------
# 7. FULL-PERIOD DETECTORS ONLY
# ------------------------------------------------------------

stable = summary[
    summary["Days_Present"] == 242
].copy()

print("\n========================================")
print("FULL 242-DAY DETECTORS")
print("========================================")

print(
    f"Stable Site-Detector pairs: "
    f"{len(stable):,}"
)

print(
    f"Unique sites represented: "
    f"{stable['NB_SCATS_SITE'].nunique():,}"
)

print(
    f"100% complete detectors: "
    f"{(stable['Complete_%'] == 100).sum():,}"
)

print(
    f"≥99% complete detectors: "
    f"{(stable['Complete_%'] >= 99).sum():,}"
)

print(
    f"Zero alarm days: "
    f"{(stable['Alarm_Days'] == 0).sum():,}"
)

print(
    f"100% positive-volume days: "
    f"{(stable['Positive_Volume_%'] == 100).sum():,}"
)

print("\n✅ Quality distribution analysis complete.")

In [ ]:
# ============================================================
# NOTEBOOK 2 — ACTIVE DETECTOR EXPLORATION
# ============================================================

import pandas as pd
import numpy as np

stable = summary[
    summary["Days_Present"] == 242
].copy()

print("========================================")
print("ACTIVE DETECTOR EXPLORATION")
print("========================================")

print(f"\nStable detectors (242 days): {len(stable):,}")
print(f"Sites represented: {stable['NB_SCATS_SITE'].nunique():,}")


# ============================================================
# 1. EXACT POSITIVE-DAY COUNTS
# ============================================================

print("\n===== POSITIVE DAY COUNTS =====")

important_days = [
    0, 1, 30, 60, 120,
    180, 194, 218, 230,
    238, 239, 240, 241, 242
]

for d in important_days:

    count = (
        stable["Positive_Volume_Days"] == d
    ).sum()

    if count > 0:
        print(
            f"Exactly {d:3d} positive days: "
            f"{count:,}"
        )


# ============================================================
# 2. POSITIVE-DAY ACTIVITY GROUPS
# ============================================================

conditions = [
    stable["Positive_Volume_Days"] == 0,

    stable["Positive_Volume_Days"].between(
        1, 119
    ),

    stable["Positive_Volume_Days"].between(
        120, 193
    ),

    stable["Positive_Volume_Days"].between(
        194, 217
    ),

    stable["Positive_Volume_Days"].between(
        218, 229
    ),

    stable["Positive_Volume_Days"].between(
        230, 237
    ),

    stable["Positive_Volume_Days"].between(
        238, 239
    ),

    stable["Positive_Volume_Days"].between(
        240, 241
    )
]

labels = [
    "0 days",
    "1–119 days (<50%)",
    "120–193 days (50–79%)",
    "194–217 days (80–89%)",
    "218–229 days (90–94%)",
    "230–237 days (95–97%)",
    "238–239 days (98–98.9%)",
    "240–241 days (≥99%)"
]

stable["Activity_Group"] = np.select(
    conditions,
    labels,
    default="Other"
)

activity_table = (
    stable["Activity_Group"]
    .value_counts()
    .reindex(labels)
    .fillna(0)
    .astype(int)
)

print("\n===== ACTIVITY GROUPS =====")

for group, count in activity_table.items():

    print(
        f"{group:<25} "
        f"{count:>7,} "
        f"({count / len(stable) * 100:6.2f}%)"
    )


# ============================================================
# 3. ACTIVE DETECTOR THRESHOLDS
# ============================================================

print("\n========================================")
print("ACTIVE-DETECTOR THRESHOLD COMPARISON")
print("========================================")

for threshold in [50, 80, 90, 95, 98, 99]:

    selected = stable[
        stable["Positive_Volume_%"] >= threshold
    ]

    print(
        f"\nPositive volume >= {threshold}%"
    )

    print(
        f"  Detectors: "
        f"{len(selected):,}"
    )

    print(
        f"  Sites: "
        f"{selected['NB_SCATS_SITE'].nunique():,}"
    )


# ============================================================
# 4. QUALITY INTERSECTION SCENARIOS
# ============================================================

print("\n========================================")
print("QUALITY SCENARIOS")
print("========================================")

print(
    "\nAll scenarios require:"
    "\n- Presence = 242 days"
    "\n- Completeness >= 99%"
    "\n- Alarm days <= 5%"
)

for positive_threshold in [90, 95, 98, 99]:

    candidate = stable[
        (stable["Complete_%"] >= 99) &
        (stable["Alarm_%"] <= 5) &
        (
            stable["Positive_Volume_%"]
            >= positive_threshold
        )
    ]

    print(
        f"\nPositive >= {positive_threshold}%:"
    )

    print(
        f"  Candidate detectors: "
        f"{len(candidate):,}"
    )

    print(
        f"  Candidate sites: "
        f"{candidate['NB_SCATS_SITE'].nunique():,}"
    )

    if len(candidate) > 0:

        print(
            f"  Median daily volume: "
            f"{candidate['Mean_Positive_Daily_Volume'].median():,.1f}"
        )


# ============================================================
# 5. TOP POSITIVE-DAY VALUES
# ============================================================

print("\n========================================")
print("MOST COMMON POSITIVE-DAY COUNTS")
print("========================================")

print(
    stable["Positive_Volume_Days"]
    .value_counts()
    .sort_values(
        ascending=False
    )
    .head(20)
)

print("\n✅ Active-detector exploration complete.")

In [ ]:
# ============================================================
# NOTEBOOK 2 — DAILY QUALITY PROFILE FOR ACTIVE DETECTORS
# ============================================================

import pandas as pd
import numpy as np
import zipfile
import re
import gc

# ------------------------------------------------------------
# 1. PROVISIONAL ACTIVE DETECTOR POOL
# ------------------------------------------------------------

active_pool = stable[
    stable["Positive_Volume_%"] >= 95
].copy()

print("========================================")
print("PROVISIONAL ACTIVE DETECTOR POOL")
print("========================================")

print(f"Active detectors: {len(active_pool):,}")
print(
    f"Sites represented: "
    f"{active_pool['NB_SCATS_SITE'].nunique():,}"
)

# Composite key: Site * 100 + Detector
candidate_keys = (
    active_pool["NB_SCATS_SITE"].astype("int64") * 100
    + active_pool["NB_DETECTOR"].astype("int64")
).to_numpy()

candidate_keys = np.unique(candidate_keys)

EXPECTED_ROWS = len(active_pool)

print(f"Expected detector rows per day: {EXPECTED_ROWS:,}")


# ------------------------------------------------------------
# 2. READ DAILY FILES AND BUILD DATE-LEVEL PROFILE
# ------------------------------------------------------------

USE_COLS_DAILY = [
    "NB_SCATS_SITE",
    "NB_DETECTOR",
    "QT_VOLUME_24HOUR",
    "CT_RECORDS",
    "CT_ALARM_24HOUR"
]

daily_results = []

print("\n========================================")
print("BUILDING DAILY QUALITY PROFILE")
print("========================================\n")

for zip_path in zip_files:

    print(f"Processing: {zip_path.name}")

    with zipfile.ZipFile(zip_path, "r") as z:

        csv_names = sorted([
            name for name in z.namelist()
            if name.lower().endswith(".csv")
        ])

        for csv_name in csv_names:

            # Extract date
            match = re.search(
                r"VSDATA_(\d{8})\.csv",
                csv_name,
                re.IGNORECASE
            )

            if not match:
                continue

            date = pd.to_datetime(
                match.group(1),
                format="%Y%m%d"
            )

            with z.open(csv_name) as f:

                df = pd.read_csv(
                    f,
                    usecols=USE_COLS_DAILY,
                    dtype={
                        "NB_SCATS_SITE": "int32",
                        "NB_DETECTOR": "int16",
                        "QT_VOLUME_24HOUR": "int32",
                        "CT_RECORDS": "int16",
                        "CT_ALARM_24HOUR": "int16"
                    }
                )

            # Build composite key
            row_keys = (
                df["NB_SCATS_SITE"].astype("int64") * 100
                + df["NB_DETECTOR"].astype("int64")
            ).to_numpy()

            # Keep provisional active detectors only
            mask = np.isin(
                row_keys,
                candidate_keys,
                assume_unique=False
            )

            d = df.loc[mask]

            total = len(d)

            positive = (
                d["QT_VOLUME_24HOUR"] > 0
            ).sum()

            zero = (
                d["QT_VOLUME_24HOUR"] == 0
            ).sum()

            negative = (
                d["QT_VOLUME_24HOUR"] < 0
            ).sum()

            incomplete = (
                d["CT_RECORDS"] < 96
            ).sum()

            alarm = (
                d["CT_ALARM_24HOUR"] > 0
            ).sum()

            alarm_intervals = (
                d["CT_ALARM_24HOUR"]
                .clip(lower=0)
                .sum()
            )

            daily_results.append({
                "Date": date,
                "Rows": total,
                "Positive": int(positive),
                "Zero": int(zero),
                "Negative": int(negative),
                "Incomplete": int(incomplete),
                "Alarm_Detectors": int(alarm),
                "Alarm_Intervals": int(alarm_intervals)
            })

            del df
            del d
            del row_keys
            del mask
            gc.collect()

    print("  ✅ Done")


# ------------------------------------------------------------
# 3. CREATE DAILY TABLE
# ------------------------------------------------------------

daily_quality = pd.DataFrame(
    daily_results
).sort_values("Date").reset_index(drop=True)

daily_quality["Positive_%"] = (
    daily_quality["Positive"]
    / daily_quality["Rows"] * 100
)

daily_quality["Zero_%"] = (
    daily_quality["Zero"]
    / daily_quality["Rows"] * 100
)

daily_quality["Negative_%"] = (
    daily_quality["Negative"]
    / daily_quality["Rows"] * 100
)

daily_quality["Alarm_Detector_%"] = (
    daily_quality["Alarm_Detectors"]
    / daily_quality["Rows"] * 100
)

daily_quality["Incomplete_%"] = (
    daily_quality["Incomplete"]
    / daily_quality["Rows"] * 100
)


# ------------------------------------------------------------
# 4. BASIC CHECK
# ------------------------------------------------------------

print("\n========================================")
print("DAILY PROFILE SUMMARY")
print("========================================")

print(f"Days analyzed: {len(daily_quality)}")

print(
    f"Date range: "
    f"{daily_quality['Date'].min().date()} → "
    f"{daily_quality['Date'].max().date()}"
)

print(
    f"Expected rows/day: "
    f"{EXPECTED_ROWS:,}"
)

print(
    f"Minimum rows/day: "
    f"{daily_quality['Rows'].min():,}"
)

print(
    f"Maximum rows/day: "
    f"{daily_quality['Rows'].max():,}"
)


# ------------------------------------------------------------
# 5. LOWEST POSITIVE-VOLUME DAYS
# ------------------------------------------------------------

print("\n========================================")
print("10 LOWEST POSITIVE-VOLUME DAYS")
print("========================================")

print(
    daily_quality[
        [
            "Date",
            "Positive",
            "Zero",
            "Negative",
            "Positive_%",
            "Alarm_Detectors",
            "Incomplete"
        ]
    ]
    .sort_values("Positive_%")
    .head(10)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 6. HIGHEST NEGATIVE-VOLUME DAYS
# ------------------------------------------------------------

print("\n========================================")
print("10 HIGHEST NEGATIVE-VOLUME DAYS")
print("========================================")

print(
    daily_quality[
        [
            "Date",
            "Negative",
            "Negative_%",
            "Alarm_Detectors",
            "Incomplete"
        ]
    ]
    .sort_values(
        "Negative_%",
        ascending=False
    )
    .head(10)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 7. HIGHEST ZERO-VOLUME DAYS
# ------------------------------------------------------------

print("\n========================================")
print("10 HIGHEST ZERO-VOLUME DAYS")
print("========================================")

print(
    daily_quality[
        [
            "Date",
            "Zero",
            "Zero_%",
            "Alarm_Detectors",
            "Incomplete"
        ]
    ]
    .sort_values(
        "Zero_%",
        ascending=False
    )
    .head(10)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 8. HIGHEST ALARM DAYS
# ------------------------------------------------------------

print("\n========================================")
print("10 HIGHEST ALARM DAYS")
print("========================================")

print(
    daily_quality[
        [
            "Date",
            "Alarm_Detectors",
            "Alarm_Detector_%",
            "Alarm_Intervals",
            "Negative",
            "Incomplete"
        ]
    ]
    .sort_values(
        "Alarm_Detector_%",
        ascending=False
    )
    .head(10)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 9. PERFECT POSITIVE DAYS
# ------------------------------------------------------------

perfect_days = (
    daily_quality["Positive"] == EXPECTED_ROWS
).sum()

print("\n========================================")
print("FULL POSITIVE-DAY CHECK")
print("========================================")

print(
    f"Days where ALL active detectors "
    f"had positive volume: {perfect_days}"
)

print(
    f"Days with at least one "
    f"zero/negative detector: "
    f"{len(daily_quality) - perfect_days}"
)


# ------------------------------------------------------------
# 10. SAVE
# ------------------------------------------------------------

daily_output = (
    OUTPUT_FOLDER /
    "active_detector_daily_quality.csv"
)

daily_quality.to_csv(
    daily_output,
    index=False
)

print("\nSaved to:")
print(daily_output)

print("\n✅ Daily anomaly investigation complete.")

In [ ]:
# ============================================================
# NOTEBOOK 2 — GLOBAL DAY-LEVEL ANOMALY DETECTION
# ============================================================

import pandas as pd
import numpy as np

dq = daily_quality.copy()

print("========================================")
print("GLOBAL DAY-LEVEL QUALITY DISTRIBUTION")
print("========================================")


# ============================================================
# 1. DISTRIBUTION OF DAILY FAILURE METRICS
# ============================================================

metrics = [
    "Negative_%",
    "Zero_%",
    "Alarm_Detector_%",
    "Incomplete_%"
]

for metric in metrics:

    print(f"\n===== {metric} =====")

    print(
        dq[metric].describe(
            percentiles=[
                0.50,
                0.75,
                0.90,
                0.95,
                0.975,
                0.99
            ]
        )
    )


# ============================================================
# 2. ROBUST OUTLIER DETECTION USING IQR
# ============================================================

def iqr_upper_limit(series, multiplier=3.0):

    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)

    iqr = q3 - q1

    upper = q3 + multiplier * iqr

    return q1, q3, iqr, upper


print("\n========================================")
print("ROBUST IQR LIMITS")
print("========================================")

limits = {}

for metric in [
    "Negative_%",
    "Alarm_Detector_%",
    "Incomplete_%"
]:

    q1, q3, iqr, upper = iqr_upper_limit(
        dq[metric],
        multiplier=3.0
    )

    limits[metric] = upper

    print(f"\n{metric}")
    print(f"Q1: {q1:.4f}")
    print(f"Q3: {q3:.4f}")
    print(f"IQR: {iqr:.4f}")
    print(f"Extreme upper limit: {upper:.4f}%")


# ============================================================
# 3. FLAG EXTREME NETWORK-LEVEL DAYS
# ============================================================

dq["Negative_Outlier"] = (
    dq["Negative_%"]
    > limits["Negative_%"]
)

dq["Alarm_Outlier"] = (
    dq["Alarm_Detector_%"]
    > limits["Alarm_Detector_%"]
)

dq["Incomplete_Outlier"] = (
    dq["Incomplete_%"]
    > limits["Incomplete_%"]
)

# Count how many quality indicators flag each day
dq["Quality_Outlier_Count"] = (
    dq[
        [
            "Negative_Outlier",
            "Alarm_Outlier",
            "Incomplete_Outlier"
        ]
    ]
    .sum(axis=1)
)

# Global anomaly if negative/alarm pattern is extreme
dq["Global_Anomaly_Day"] = (
    dq["Negative_Outlier"] |
    dq["Alarm_Outlier"]
)


# ============================================================
# 4. RESULTS
# ============================================================

global_anomalies = (
    dq[
        dq["Global_Anomaly_Day"]
    ]
    .copy()
    .sort_values(
        "Negative_%",
        ascending=False
    )
)

print("\n========================================")
print("GLOBAL ANOMALY DAYS")
print("========================================")

print(
    f"Detected anomaly days: "
    f"{len(global_anomalies)} / {len(dq)}"
)

print(
    f"Percentage of study period: "
    f"{len(global_anomalies) / len(dq) * 100:.2f}%"
)

print("\nDetected dates:\n")

print(
    global_anomalies[
        [
            "Date",
            "Positive_%",
            "Negative_%",
            "Zero_%",
            "Alarm_Detector_%",
            "Incomplete_%",
            "Quality_Outlier_Count"
        ]
    ].to_string(index=False)
)


# ============================================================
# 5. COMPARE NORMAL VS ANOMALY DAYS
# ============================================================

normal_days = dq[
    ~dq["Global_Anomaly_Day"]
]

print("\n========================================")
print("NORMAL DAYS VS GLOBAL ANOMALY DAYS")
print("========================================")

comparison = pd.DataFrame({

    "Normal_Days": [
        len(normal_days),
        normal_days["Positive_%"].median(),
        normal_days["Negative_%"].median(),
        normal_days["Alarm_Detector_%"].median(),
        normal_days["Incomplete_%"].median()
    ],

    "Global_Anomaly_Days": [
        len(global_anomalies),
        global_anomalies["Positive_%"].median(),
        global_anomalies["Negative_%"].median(),
        global_anomalies["Alarm_Detector_%"].median(),
        global_anomalies["Incomplete_%"].median()
    ]

}, index=[
    "Number_of_Days",
    "Median_Positive_%",
    "Median_Negative_%",
    "Median_Alarm_%",
    "Median_Incomplete_%"
])

print(comparison)


# ============================================================
# 6. SAVE
# ============================================================

dq.to_csv(
    OUTPUT_FOLDER /
    "daily_quality_with_global_anomalies.csv",
    index=False
)

global_anomalies.to_csv(
    OUTPUT_FOLDER /
    "global_anomaly_days.csv",
    index=False
)

print("\n✅ Global anomaly detection complete.")

In [ ]:
# ============================================================
# NOTEBOOK 2 — DAY SEVERITY CLASSIFICATION
# ============================================================

import pandas as pd
import numpy as np

dq2 = dq.copy()

# ------------------------------------------------------------
# 1. MAJOR SYSTEM-WIDE DISRUPTION
# ------------------------------------------------------------

MAJOR_THRESHOLD = 10.0   # percentage of candidate detectors

dq2["Major_System_Disruption"] = (
    (dq2["Negative_%"] >= MAJOR_THRESHOLD) |
    (dq2["Alarm_Detector_%"] >= MAJOR_THRESHOLD)
)

# ------------------------------------------------------------
# 2. ELEVATED QUALITY DAY
#    IQR anomaly, but not major system disruption
# ------------------------------------------------------------

dq2["Elevated_Quality_Day"] = (
    dq2["Global_Anomaly_Day"] &
    ~dq2["Major_System_Disruption"]
)

# ------------------------------------------------------------
# 3. FINAL DAY CLASS
# ------------------------------------------------------------

dq2["Day_Class"] = np.select(
    [
        dq2["Major_System_Disruption"],
        dq2["Elevated_Quality_Day"]
    ],
    [
        "Major system disruption",
        "Elevated quality risk"
    ],
    default="Normal"
)

# ------------------------------------------------------------
# 4. SUMMARY
# ------------------------------------------------------------

print("========================================")
print("DAY-SEVERITY CLASSIFICATION")
print("========================================")

class_counts = dq2["Day_Class"].value_counts()

for cls, count in class_counts.items():

    print(
        f"{cls:<28}: "
        f"{count:3d} days "
        f"({count / len(dq2) * 100:5.2f}%)"
    )


# ------------------------------------------------------------
# 5. MAJOR DAYS
# ------------------------------------------------------------

major_days = dq2[
    dq2["Major_System_Disruption"]
].copy()

print("\n========================================")
print("MAJOR SYSTEM-DISRUPTION DAYS")
print("========================================")

print(
    major_days[
        [
            "Date",
            "Positive_%",
            "Negative_%",
            "Alarm_Detector_%",
            "Incomplete_%"
        ]
    ]
    .sort_values("Date")
    .to_string(index=False)
)


# ------------------------------------------------------------
# 6. ELEVATED BUT NOT MAJOR
# ------------------------------------------------------------

elevated_days = dq2[
    dq2["Elevated_Quality_Day"]
].copy()

print("\n========================================")
print("ELEVATED QUALITY-RISK DAYS")
print("========================================")

print(
    elevated_days[
        [
            "Date",
            "Positive_%",
            "Negative_%",
            "Alarm_Detector_%",
            "Incomplete_%"
        ]
    ]
    .sort_values("Date")
    .to_string(index=False)
)


# ------------------------------------------------------------
# 7. MODELING-DAY COUNTS
# ------------------------------------------------------------

usable_for_activity = dq2[
    ~dq2["Major_System_Disruption"]
]

print("\n========================================")
print("ACTIVITY-ELIGIBILITY PERIOD")
print("========================================")

print(f"Original study days: {len(dq2)}")

print(
    f"Major system-disruption days removed: "
    f"{len(major_days)}"
)

print(
    f"Days used to assess detector activity: "
    f"{len(usable_for_activity)}"
)


# ------------------------------------------------------------
# 8. SAVE
# ------------------------------------------------------------

dq2.to_csv(
    OUTPUT_FOLDER /
    "daily_quality_classified.csv",
    index=False
)

major_days.to_csv(
    OUTPUT_FOLDER /
    "major_system_disruption_days.csv",
    index=False
)

print("\n✅ Day classification saved.")

In [ ]:
# ============================================================
# NOTEBOOK 2 — RECALCULATE DETECTOR ACTIVITY
# EXCLUDING MAJOR SYSTEM-DISRUPTION DAYS
# ============================================================

import pandas as pd
import numpy as np
import zipfile
import re
import gc

# ------------------------------------------------------------
# 1. STABLE DETECTORS
# ------------------------------------------------------------

stable_recalc = summary[
    summary["Days_Present"] == 242
][
    ["NB_SCATS_SITE", "NB_DETECTOR"]
].copy()

# Unique composite key
stable_recalc["Key"] = (
    stable_recalc["NB_SCATS_SITE"].astype("int64") * 100
    + stable_recalc["NB_DETECTOR"].astype("int64")
)

stable_recalc = (
    stable_recalc
    .sort_values("Key")
    .reset_index(drop=True)
)

stable_keys = stable_recalc["Key"].to_numpy()

N = len(stable_recalc)

print("========================================")
print("RECALCULATING DETECTOR ACTIVITY")
print("========================================")

print(f"Stable detectors: {N:,}")


# ------------------------------------------------------------
# 2. MAJOR-DISRUPTION DATES
# ------------------------------------------------------------

major_date_set = set(
    pd.to_datetime(
        major_days["Date"]
    ).dt.strftime("%Y%m%d")
)

print(f"Major disruption days excluded: {len(major_date_set)}")

print("\nExcluded dates:")
for d in sorted(major_date_set):
    print(d)


# ------------------------------------------------------------
# 3. ACCUMULATORS
# ------------------------------------------------------------

positive_days = np.zeros(N, dtype=np.int16)
zero_days = np.zeros(N, dtype=np.int16)
negative_days = np.zeros(N, dtype=np.int16)

complete_days = np.zeros(N, dtype=np.int16)
alarm_days = np.zeros(N, dtype=np.int16)

positive_volume_sum = np.zeros(N, dtype=np.int64)

eligible_days_processed = 0
match_errors = []


USE_COLS_RECALC = [
    "NB_SCATS_SITE",
    "NB_DETECTOR",
    "QT_VOLUME_24HOUR",
    "CT_RECORDS",
    "CT_ALARM_24HOUR"
]


# ------------------------------------------------------------
# 4. STREAM THROUGH ALL DAILY FILES
# ------------------------------------------------------------

print("\n========================================")
print("PROCESSING 234 ELIGIBLE DAYS")
print("========================================\n")

for zip_path in zip_files:

    print(f"Processing: {zip_path.name}")

    with zipfile.ZipFile(zip_path, "r") as z:

        csv_names = sorted([
            name for name in z.namelist()
            if name.lower().endswith(".csv")
        ])

        for csv_name in csv_names:

            match_date = re.search(
                r"VSDATA_(\d{8})\.csv",
                csv_name,
                re.IGNORECASE
            )

            if not match_date:
                continue

            date_string = match_date.group(1)

            # Skip major system-disruption days
            if date_string in major_date_set:
                continue

            with z.open(csv_name) as f:

                df = pd.read_csv(
                    f,
                    usecols=USE_COLS_RECALC,
                    dtype={
                        "NB_SCATS_SITE": "int32",
                        "NB_DETECTOR": "int16",
                        "QT_VOLUME_24HOUR": "int32",
                        "CT_RECORDS": "int16",
                        "CT_ALARM_24HOUR": "int16"
                    }
                )

            row_keys = (
                df["NB_SCATS_SITE"].astype("int64") * 100
                + df["NB_DETECTOR"].astype("int64")
            ).to_numpy()

            # Find stable detector rows efficiently
            positions = np.searchsorted(
                stable_keys,
                row_keys
            )

            valid = positions < N

            matched = np.zeros(
                len(row_keys),
                dtype=bool
            )

            matched[valid] = (
                stable_keys[positions[valid]]
                == row_keys[valid]
            )

            stable_positions = positions[matched]

            d = df.loc[
                matched,
                [
                    "QT_VOLUME_24HOUR",
                    "CT_RECORDS",
                    "CT_ALARM_24HOUR"
                ]
            ]

            # Safety check
            if len(d) != N:

                match_errors.append(
                    (date_string, len(d))
                )

            volume = (
                d["QT_VOLUME_24HOUR"]
                .to_numpy()
            )

            records = (
                d["CT_RECORDS"]
                .to_numpy()
            )

            alarms = (
                d["CT_ALARM_24HOUR"]
                .to_numpy()
            )

            # ------------------------------------------------
            # ACCUMULATE
            # ------------------------------------------------

            positive_mask = volume > 0
            zero_mask = volume == 0
            negative_mask = volume < 0

            positive_days[stable_positions] += (
                positive_mask.astype(np.int16)
            )

            zero_days[stable_positions] += (
                zero_mask.astype(np.int16)
            )

            negative_days[stable_positions] += (
                negative_mask.astype(np.int16)
            )

            complete_days[stable_positions] += (
                (records == 96).astype(np.int16)
            )

            alarm_days[stable_positions] += (
                (alarms > 0).astype(np.int16)
            )

            positive_volume_sum[stable_positions] += np.where(
                positive_mask,
                volume,
                0
            ).astype(np.int64)

            eligible_days_processed += 1

            del df
            del d
            del row_keys
            del positions
            del matched
            gc.collect()

    print("  ✅ Done")


# ------------------------------------------------------------
# 5. VALIDATION
# ------------------------------------------------------------

print("\n========================================")
print("PROCESSING VALIDATION")
print("========================================")

print(
    f"Eligible days processed: "
    f"{eligible_days_processed}"
)

print(
    f"Expected eligible days: "
    f"{242 - len(major_date_set)}"
)

print(
    f"Days with detector-count mismatch: "
    f"{len(match_errors)}"
)

if match_errors:
    print(match_errors[:10])


# ------------------------------------------------------------
# 6. CREATE RECALCULATED SUMMARY
# ------------------------------------------------------------

ELIGIBLE_DAYS = eligible_days_processed

stable_recalc[
    "Eligible_Days"
] = ELIGIBLE_DAYS

stable_recalc[
    "Positive_Eligible_Days"
] = positive_days

stable_recalc[
    "Zero_Eligible_Days"
] = zero_days

stable_recalc[
    "Negative_Eligible_Days"
] = negative_days

stable_recalc[
    "Complete_Eligible_Days"
] = complete_days

stable_recalc[
    "Alarm_Eligible_Days"
] = alarm_days


stable_recalc[
    "Positive_Eligible_%"
] = (
    positive_days
    / ELIGIBLE_DAYS
    * 100
)

stable_recalc[
    "Complete_Eligible_%"
] = (
    complete_days
    / ELIGIBLE_DAYS
    * 100
)

stable_recalc[
    "Alarm_Eligible_%"
] = (
    alarm_days
    / ELIGIBLE_DAYS
    * 100
)

stable_recalc[
    "Mean_Positive_Daily_Volume"
] = np.where(
    positive_days > 0,
    positive_volume_sum
    / positive_days,
    0
)


# ------------------------------------------------------------
# 7. ACTIVITY THRESHOLD RESULTS
# ------------------------------------------------------------

print("\n========================================")
print("RECALCULATED ACTIVITY THRESHOLDS")
print("========================================")

for threshold in [
    80, 90, 95, 98, 99
]:

    selected = stable_recalc[
        stable_recalc[
            "Positive_Eligible_%"
        ] >= threshold
    ]

    print(
        f"\nPositive on >= {threshold}% "
        f"of eligible days:"
    )

    print(
        f"  Detectors: "
        f"{len(selected):,}"
    )

    print(
        f"  Sites: "
        f"{selected['NB_SCATS_SITE'].nunique():,}"
    )


# ------------------------------------------------------------
# 8. FINAL ACTIVE POOL — PROVISIONAL
# ------------------------------------------------------------

final_active = stable_recalc[
    stable_recalc[
        "Positive_Eligible_%"
    ] >= 95
].copy()

print("\n========================================")
print("FINAL ACTIVE POOL — ACTIVITY CRITERION")
print("========================================")

print(
    f"Active detectors: "
    f"{len(final_active):,}"
)

print(
    f"Sites represented: "
    f"{final_active['NB_SCATS_SITE'].nunique():,}"
)

print(
    f"Median positive-day percentage: "
    f"{final_active['Positive_Eligible_%'].median():.2f}%"
)

print(
    f"Median completeness: "
    f"{final_active['Complete_Eligible_%'].median():.2f}%"
)

print(
    f"Median alarm-day percentage: "
    f"{final_active['Alarm_Eligible_%'].median():.2f}%"
)

print(
    f"Median positive daily volume: "
    f"{final_active['Mean_Positive_Daily_Volume'].median():,.1f}"
)


# ------------------------------------------------------------
# 9. SAVE
# ------------------------------------------------------------

stable_recalc.to_csv(
    OUTPUT_FOLDER /
    "stable_detector_activity_recalculated.csv",
    index=False
)

final_active.to_csv(
    OUTPUT_FOLDER /
    "final_active_detector_pool_activity_only.csv",
    index=False
)

print("\n✅ Recalculation complete.")

In [ ]:
# ============================================================
# NOTEBOOK 2 — FINAL ACTIVE-POOL QUALITY AUDIT
# ============================================================

import pandas as pd
import numpy as np

fa = final_active.copy()

print("========================================")
print("FINAL ACTIVE-POOL QUALITY AUDIT")
print("========================================")

print(f"\nActive detectors: {len(fa):,}")
print(
    f"Sites represented: "
    f"{fa['NB_SCATS_SITE'].nunique():,}"
)


# ============================================================
# 1. ACTIVITY DISTRIBUTION
# ============================================================

print("\n========================================")
print("ACTIVITY DISTRIBUTION")
print("========================================")

print(
    fa["Positive_Eligible_%"].describe(
        percentiles=[
            0.01, 0.05, 0.10,
            0.25, 0.50, 0.75,
            0.90, 0.95, 0.99
        ]
    )
)


# ============================================================
# 2. COMPLETENESS DISTRIBUTION
# ============================================================

print("\n========================================")
print("COMPLETENESS DISTRIBUTION")
print("========================================")

print(
    fa["Complete_Eligible_%"].describe(
        percentiles=[
            0.01, 0.05, 0.10,
            0.25, 0.50, 0.75,
            0.90, 0.95, 0.99
        ]
    )
)

print("\nCompleteness thresholds:")

for threshold in [100, 99.5, 99, 98, 95]:

    n = (
        fa["Complete_Eligible_%"]
        >= threshold
    ).sum()

    print(
        f">= {threshold:5.1f}% : "
        f"{n:,} "
        f"({n / len(fa) * 100:.2f}%)"
    )


# ============================================================
# 3. ALARM DISTRIBUTION
# ============================================================

print("\n========================================")
print("ALARM-DAY DISTRIBUTION")
print("========================================")

print(
    fa["Alarm_Eligible_%"].describe(
        percentiles=[
            0.01, 0.05, 0.10,
            0.25, 0.50, 0.75,
            0.90, 0.95, 0.99
        ]
    )
)

print("\nAlarm thresholds:")

for threshold in [1, 2, 5, 10, 20]:

    n = (
        fa["Alarm_Eligible_%"]
        <= threshold
    ).sum()

    print(
        f"<= {threshold:2d}% : "
        f"{n:,} "
        f"({n / len(fa) * 100:.2f}%)"
    )


# ============================================================
# 4. ZERO / NEGATIVE ELIGIBLE DAYS
# ============================================================

print("\n========================================")
print("ZERO / NEGATIVE DAYS")
print("========================================")

print(
    f"Detectors with >=1 zero day: "
    f"{(fa['Zero_Eligible_Days'] > 0).sum():,}"
)

print(
    f"Detectors with >=1 negative day: "
    f"{(fa['Negative_Eligible_Days'] > 0).sum():,}"
)

print("\nNegative eligible days distribution:")

print(
    fa["Negative_Eligible_Days"].describe(
        percentiles=[
            0.50, 0.75,
            0.90, 0.95, 0.99
        ]
    )
)

print("\nZero eligible days distribution:")

print(
    fa["Zero_Eligible_Days"].describe(
        percentiles=[
            0.50, 0.75,
            0.90, 0.95, 0.99
        ]
    )
)


# ============================================================
# 5. TRAFFIC VOLUME DISTRIBUTION
# ============================================================

print("\n========================================")
print("MEAN POSITIVE DAILY VOLUME")
print("========================================")

print(
    fa["Mean_Positive_Daily_Volume"].describe(
        percentiles=[
            0.01, 0.05, 0.10,
            0.25, 0.50, 0.75,
            0.90, 0.95, 0.99
        ]
    )
)

print("\nLow-volume detector counts:")

for threshold in [50, 100, 250, 500]:

    n = (
        fa["Mean_Positive_Daily_Volume"]
        < threshold
    ).sum()

    print(
        f"< {threshold:4d} veh/day: "
        f"{n:,} "
        f"({n / len(fa) * 100:.2f}%)"
    )


# ============================================================
# 6. ACTIVE DETECTORS PER SITE
# ============================================================

site_counts = (
    fa.groupby("NB_SCATS_SITE")
    .size()
    .rename("Active_Detectors")
    .reset_index()
)

print("\n========================================")
print("ACTIVE DETECTORS PER SITE")
print("========================================")

print(
    site_counts["Active_Detectors"].describe(
        percentiles=[
            0.10, 0.25, 0.50,
            0.75, 0.90, 0.95, 0.99
        ]
    )
)

print("\nSites meeting detector-count thresholds:")

for threshold in [1, 2, 4, 6, 8, 10, 12]:

    n = (
        site_counts["Active_Detectors"]
        >= threshold
    ).sum()

    print(
        f">= {threshold:2d} active detectors: "
        f"{n:,} sites "
        f"({n / len(site_counts) * 100:.2f}%)"
    )


# ============================================================
# 7. MOST COMMON ACTIVE-DETECTOR COUNTS
# ============================================================

print("\n========================================")
print("MOST COMMON DETECTOR COUNTS PER SITE")
print("========================================")

print(
    site_counts["Active_Detectors"]
    .value_counts()
    .sort_index()
    .head(25)
)


# ============================================================
# 8. CROSS-CHECK HIGH-QUALITY SUBSETS
# ============================================================

print("\n========================================")
print("HIGH-QUALITY SUBSET COMPARISON")
print("========================================")

scenarios = {
    "Activity only (>=95%)":
        np.ones(len(fa), dtype=bool),

    "Activity + Complete >=98%":
        fa["Complete_Eligible_%"] >= 98,

    "Activity + Complete >=99%":
        fa["Complete_Eligible_%"] >= 99,

    "Activity + Complete >=99.5%":
        fa["Complete_Eligible_%"] >= 99.5,

    "Activity + Complete >=99% + Alarm <=5%":
        (
            (fa["Complete_Eligible_%"] >= 99) &
            (fa["Alarm_Eligible_%"] <= 5)
        )
}

for name, mask in scenarios.items():

    temp = fa[mask]

    print(f"\n{name}")

    print(
        f"  Detectors: "
        f"{len(temp):,}"
    )

    print(
        f"  Sites: "
        f"{temp['NB_SCATS_SITE'].nunique():,}"
    )


# ============================================================
# 9. SAVE SITE SUMMARY
# ============================================================

site_counts.to_csv(
    OUTPUT_FOLDER /
    "final_active_detectors_per_site.csv",
    index=False
)

print("\n✅ Final active-pool quality audit complete.")

In [ ]:
# ============================================================
# NOTEBOOK 2 — FREEZE FINAL DETECTOR POOL
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# FINAL SELECTION CRITERIA
# ------------------------------------------------------------

FINAL_ACTIVITY_THRESHOLD = 95.0
FINAL_COMPLETENESS_THRESHOLD = 99.0

final_pool = stable_recalc[
    (stable_recalc["Positive_Eligible_%"] >= FINAL_ACTIVITY_THRESHOLD) &
    (stable_recalc["Complete_Eligible_%"] >= FINAL_COMPLETENESS_THRESHOLD)
].copy()

# ------------------------------------------------------------
# ADD UNIQUE DETECTOR ID
# ------------------------------------------------------------

final_pool["Detector_ID"] = (
    final_pool["NB_SCATS_SITE"].astype(str)
    + "_"
    + final_pool["NB_DETECTOR"].astype(str)
)

# ------------------------------------------------------------
# QUALITY LABELS — descriptive only
# ------------------------------------------------------------

final_pool["Volume_Group"] = pd.cut(
    final_pool["Mean_Positive_Daily_Volume"],
    bins=[-np.inf, 100, 250, 500, 2000, 5000, np.inf],
    labels=[
        "<100",
        "100–249",
        "250–499",
        "500–1999",
        "2000–4999",
        "5000+"
    ],
    right=False
)

# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

print("========================================")
print("FINAL DETECTOR POOL")
print("========================================")

print(f"Final detectors: {len(final_pool):,}")

print(
    f"Final sites: "
    f"{final_pool['NB_SCATS_SITE'].nunique():,}"
)

print(
    f"Activity threshold: "
    f">= {FINAL_ACTIVITY_THRESHOLD}%"
)

print(
    f"Completeness threshold: "
    f">= {FINAL_COMPLETENESS_THRESHOLD}%"
)

print(
    f"Median activity: "
    f"{final_pool['Positive_Eligible_%'].median():.2f}%"
)

print(
    f"Median completeness: "
    f"{final_pool['Complete_Eligible_%'].median():.2f}%"
)

print(
    f"Median alarm-day rate: "
    f"{final_pool['Alarm_Eligible_%'].median():.2f}%"
)

print(
    f"Median daily volume: "
    f"{final_pool['Mean_Positive_Daily_Volume'].median():,.1f}"
)


# ------------------------------------------------------------
# VOLUME GROUP DISTRIBUTION
# ------------------------------------------------------------

print("\n========================================")
print("VOLUME GROUPS")
print("========================================")

volume_counts = (
    final_pool["Volume_Group"]
    .value_counts(sort=False)
)

for group, count in volume_counts.items():

    print(
        f"{str(group):<12}: "
        f"{count:>6,} "
        f"({count / len(final_pool) * 100:5.2f}%)"
    )


# ------------------------------------------------------------
# ALARM EXPOSURE
# ------------------------------------------------------------

print("\n========================================")
print("REAL ALARM EXPOSURE RETAINED")
print("========================================")

with_alarm = (
    final_pool["Alarm_Eligible_Days"] > 0
).sum()

without_alarm = (
    final_pool["Alarm_Eligible_Days"] == 0
).sum()

print(
    f"Detectors with >=1 real alarm day: "
    f"{with_alarm:,} "
    f"({with_alarm / len(final_pool) * 100:.2f}%)"
)

print(
    f"Detectors with no alarm days: "
    f"{without_alarm:,} "
    f"({without_alarm / len(final_pool) * 100:.2f}%)"
)

print(
    f"Total alarm-affected detector-days: "
    f"{final_pool['Alarm_Eligible_Days'].sum():,}"
)


# ------------------------------------------------------------
# SAVE FINAL POOL
# ------------------------------------------------------------

FINAL_POOL_FILE = (
    OUTPUT_FOLDER /
    "FINAL_detector_pool.csv"
)

final_pool.to_csv(
    FINAL_POOL_FILE,
    index=False
)

print("\n========================================")
print("SAVED")
print("========================================")

print(FINAL_POOL_FILE)

print("\n✅ Final detector pool frozen successfully.")

In [ ]:
# ============================================================
# NOTEBOOK 2 — ADD REGION INFORMATION TO FINAL POOL
# ============================================================

import pandas as pd
import numpy as np
import zipfile
import re

print("========================================")
print("ADDING REGION INFORMATION")
print("========================================")


# ------------------------------------------------------------
# 1. FIND EARLIEST DAILY CSV
# ------------------------------------------------------------

daily_file_records = []

for zip_path in zip_files:

    with zipfile.ZipFile(zip_path, "r") as z:

        for name in z.namelist():

            match = re.search(
                r"VSDATA_(\d{8})\.csv",
                name,
                re.IGNORECASE
            )

            if match:

                daily_file_records.append({
                    "date": match.group(1),
                    "zip_path": zip_path,
                    "csv_name": name
                })


earliest = min(
    daily_file_records,
    key=lambda x: x["date"]
)

print(
    f"Earliest daily file: "
    f"{earliest['date']}"
)

print(
    f"ZIP source: "
    f"{earliest['zip_path'].name}"
)


# ------------------------------------------------------------
# 2. READ SITE / DETECTOR / REGION ONLY
# ------------------------------------------------------------

with zipfile.ZipFile(
    earliest["zip_path"],
    "r"
) as z:

    with z.open(
        earliest["csv_name"]
    ) as f:

        region_df = pd.read_csv(
            f,
            usecols=[
                "NB_SCATS_SITE",
                "NB_DETECTOR",
                "NM_REGION"
            ],
            dtype={
                "NB_SCATS_SITE": "int32",
                "NB_DETECTOR": "int16",
                "NM_REGION": "string"
            }
        )


print(
    f"\nRows in reference day: "
    f"{len(region_df):,}"
)


# ------------------------------------------------------------
# 3. CHECK DUPLICATES
# ------------------------------------------------------------

duplicate_region_keys = (
    region_df.duplicated(
        subset=[
            "NB_SCATS_SITE",
            "NB_DETECTOR"
        ]
    )
    .sum()
)

print(
    f"Duplicate Site-Detector keys: "
    f"{duplicate_region_keys:,}"
)


# ------------------------------------------------------------
# 4. MERGE WITH FINAL POOL
# ------------------------------------------------------------

final_pool_region = final_pool.merge(
    region_df,
    on=[
        "NB_SCATS_SITE",
        "NB_DETECTOR"
    ],
    how="left",
    validate="one_to_one"
)


# ------------------------------------------------------------
# 5. VALIDATION
# ------------------------------------------------------------

print("\n========================================")
print("REGION MERGE VALIDATION")
print("========================================")

print(
    f"Final pool before merge: "
    f"{len(final_pool):,}"
)

print(
    f"Final pool after merge: "
    f"{len(final_pool_region):,}"
)

missing_region = (
    final_pool_region["NM_REGION"]
    .isna()
    .sum()
)

print(
    f"Missing region values: "
    f"{missing_region:,}"
)

print(
    f"Unique regions: "
    f"{final_pool_region['NM_REGION'].nunique()}"
)


# ------------------------------------------------------------
# 6. DETECTOR DISTRIBUTION BY REGION
# ------------------------------------------------------------

region_detector_counts = (
    final_pool_region
    .groupby("NM_REGION")
    .size()
    .rename("Detectors")
    .sort_values(
        ascending=False
    )
)

print("\n========================================")
print("DETECTORS BY REGION")
print("========================================")

print(
    region_detector_counts
    .to_string()
)


# ------------------------------------------------------------
# 7. SITE DISTRIBUTION BY REGION
# ------------------------------------------------------------

region_site_counts = (
    final_pool_region[
        [
            "NB_SCATS_SITE",
            "NM_REGION"
        ]
    ]
    .drop_duplicates()
    .groupby("NM_REGION")
    .size()
    .rename("Sites")
    .sort_values(
        ascending=False
    )
)

print("\n========================================")
print("SITES BY REGION")
print("========================================")

print(
    region_site_counts
    .to_string()
)


# ------------------------------------------------------------
# 8. COMBINED REGION SUMMARY
# ------------------------------------------------------------

region_summary = pd.concat(
    [
        region_detector_counts,
        region_site_counts
    ],
    axis=1
).fillna(0)

region_summary["Detector_%"] = (
    region_summary["Detectors"]
    / len(final_pool_region)
    * 100
)

region_summary = (
    region_summary
    .sort_values(
        "Detectors",
        ascending=False
    )
)

print("\n========================================")
print("REGION SUMMARY")
print("========================================")

print(
    region_summary
    .round(2)
    .to_string()
)


# ------------------------------------------------------------
# 9. SAVE
# ------------------------------------------------------------

final_pool_region.to_csv(
    OUTPUT_FOLDER /
    "FINAL_detector_pool_with_region.csv",
    index=False
)

region_summary.to_csv(
    OUTPUT_FOLDER /
    "final_pool_region_summary.csv"
)

print("\n✅ Region information added and saved.")

In [ ]:
# ============================================================
# NOTEBOOK 2 — REGION COVERAGE CROSS-CHECK
# ============================================================

import pandas as pd
import zipfile

all_raw_regions = set()

print("========================================")
print("SCANNING RAW REGION COVERAGE")
print("========================================\n")

for zip_path in zip_files:

    month_regions = set()

    with zipfile.ZipFile(zip_path, "r") as z:

        csv_names = sorted([
            name for name in z.namelist()
            if name.lower().endswith(".csv")
        ])

        for csv_name in csv_names:

            with z.open(csv_name) as f:

                r = pd.read_csv(
                    f,
                    usecols=["NM_REGION"],
                    dtype={"NM_REGION": "string"}
                )

            regions = set(
                r["NM_REGION"]
                .dropna()
                .unique()
            )

            month_regions.update(regions)
            all_raw_regions.update(regions)

    print(
        f"{zip_path.name}: "
        f"{len(month_regions)} regions"
    )


# ------------------------------------------------------------
# FINAL POOL REGIONS
# ------------------------------------------------------------

final_regions = set(
    final_pool_region["NM_REGION"]
    .dropna()
    .unique()
)


# ------------------------------------------------------------
# COMPARISON
# ------------------------------------------------------------

missing_from_final = (
    all_raw_regions - final_regions
)

unexpected_final = (
    final_regions - all_raw_regions
)

print("\n========================================")
print("REGION COVERAGE COMPARISON")
print("========================================")

print(
    f"Regions in raw 8-month dataset: "
    f"{len(all_raw_regions)}"
)

print(
    f"Regions in final detector pool: "
    f"{len(final_regions)}"
)

print("\nRegions present in RAW data but absent from final pool:")

if missing_from_final:

    for region in sorted(missing_from_final):
        print(f"  {region}")

else:
    print("  None")


print("\nUnexpected final-pool regions:")

if unexpected_final:

    for region in sorted(unexpected_final):
        print(f"  {region}")

else:
    print("  None")

print("\n✅ Region coverage cross-check complete.")

In [ ]:
# ============================================================
# NOTEBOOK 2 — WHY WAS FT1 EXCLUDED?
# ============================================================

import pandas as pd

# Add region to all 117,160 stable detectors
stable_with_region = stable_recalc.merge(
    region_df[
        [
            "NB_SCATS_SITE",
            "NB_DETECTOR",
            "NM_REGION"
        ]
    ],
    on=[
        "NB_SCATS_SITE",
        "NB_DETECTOR"
    ],
    how="left",
    validate="one_to_one"
)

ft1 = stable_with_region[
    stable_with_region["NM_REGION"] == "FT1"
].copy()

print("========================================")
print("FT1 EXCLUSION AUDIT")
print("========================================")

print(f"\nStable FT1 detectors: {len(ft1):,}")
print(
    f"FT1 sites: "
    f"{ft1['NB_SCATS_SITE'].nunique():,}"
)

# ------------------------------------------------------------
# ACTIVITY
# ------------------------------------------------------------

print("\n===== ACTIVITY =====")

print(
    ft1["Positive_Eligible_%"].describe(
        percentiles=[
            0.10, 0.25, 0.50,
            0.75, 0.90, 0.95, 0.99
        ]
    )
)

activity_pass = (
    ft1["Positive_Eligible_%"] >= 95
).sum()

print(
    f"\nActivity >=95%: "
    f"{activity_pass:,} / {len(ft1):,}"
)


# ------------------------------------------------------------
# COMPLETENESS
# ------------------------------------------------------------

print("\n===== COMPLETENESS =====")

print(
    ft1["Complete_Eligible_%"].describe(
        percentiles=[
            0.10, 0.25, 0.50,
            0.75, 0.90, 0.95, 0.99
        ]
    )
)

complete_pass = (
    ft1["Complete_Eligible_%"] >= 99
).sum()

print(
    f"\nCompleteness >=99%: "
    f"{complete_pass:,} / {len(ft1):,}"
)


# ------------------------------------------------------------
# FINAL CRITERIA INTERSECTION
# ------------------------------------------------------------

both_pass = (
    (ft1["Positive_Eligible_%"] >= 95) &
    (ft1["Complete_Eligible_%"] >= 99)
).sum()

print("\n========================================")
print("FINAL CRITERIA")
print("========================================")

print(
    f"Activity >=95% AND Completeness >=99%: "
    f"{both_pass:,}"
)


# ------------------------------------------------------------
# WHY EACH DETECTOR FAILED
# ------------------------------------------------------------

ft1["Fails_Activity"] = (
    ft1["Positive_Eligible_%"] < 95
)

ft1["Fails_Completeness"] = (
    ft1["Complete_Eligible_%"] < 99
)

failure_table = pd.DataFrame({
    "Category": [
        "Fails activity only",
        "Fails completeness only",
        "Fails both",
        "Passes both"
    ],

    "Count": [
        (
            ft1["Fails_Activity"] &
            ~ft1["Fails_Completeness"]
        ).sum(),

        (
            ~ft1["Fails_Activity"] &
            ft1["Fails_Completeness"]
        ).sum(),

        (
            ft1["Fails_Activity"] &
            ft1["Fails_Completeness"]
        ).sum(),

        (
            ~ft1["Fails_Activity"] &
            ~ft1["Fails_Completeness"]
        ).sum()
    ]
})

print("\n===== FAILURE REASONS =====")
print(
    failure_table.to_string(index=False)
)

print("\n✅ FT1 audit complete.")

In [ ]:
# ============================================================
# NOTEBOOK 2 — FT1 SYSTEMATIC INCOMPLETENESS INVESTIGATION
# ============================================================

import pandas as pd
import numpy as np
import zipfile
import re
import gc

# ------------------------------------------------------------
# 1. FT1 DETECTORS THAT PASS THE ACTIVITY CRITERION
# ------------------------------------------------------------

ft1_active = ft1[
    ft1["Positive_Eligible_%"] >= 95
].copy()

ft1_keys = np.unique(
    (
        ft1_active["NB_SCATS_SITE"].astype("int64") * 100
        + ft1_active["NB_DETECTOR"].astype("int64")
    ).to_numpy()
)

print("========================================")
print("FT1 INCOMPLETENESS INVESTIGATION")
print("========================================")

print(f"\nFT1 active detectors: {len(ft1_active):,}")
print(
    f"FT1 active sites: "
    f"{ft1_active['NB_SCATS_SITE'].nunique():,}"
)


# ------------------------------------------------------------
# 2. PROCESS ELIGIBLE DAYS
# ------------------------------------------------------------

ft1_daily = []

USE_COLS_FT1 = [
    "NB_SCATS_SITE",
    "NB_DETECTOR",
    "QT_VOLUME_24HOUR",
    "CT_RECORDS",
    "CT_ALARM_24HOUR"
]

for zip_path in zip_files:

    print(f"Processing: {zip_path.name}")

    with zipfile.ZipFile(zip_path, "r") as z:

        csv_names = sorted([
            name for name in z.namelist()
            if name.lower().endswith(".csv")
        ])

        for csv_name in csv_names:

            match = re.search(
                r"VSDATA_(\d{8})\.csv",
                csv_name,
                re.IGNORECASE
            )

            if not match:
                continue

            date_string = match.group(1)

            # Same 8 major system-wide disruption days
            if date_string in major_date_set:
                continue

            with z.open(csv_name) as f:

                d = pd.read_csv(
                    f,
                    usecols=USE_COLS_FT1,
                    dtype={
                        "NB_SCATS_SITE": "int32",
                        "NB_DETECTOR": "int16",
                        "QT_VOLUME_24HOUR": "int32",
                        "CT_RECORDS": "int16",
                        "CT_ALARM_24HOUR": "int16"
                    }
                )

            row_keys = (
                d["NB_SCATS_SITE"].astype("int64") * 100
                + d["NB_DETECTOR"].astype("int64")
            ).to_numpy()

            mask = np.isin(
                row_keys,
                ft1_keys
            )

            x = d.loc[mask].copy()

            incomplete = x[
                x["CT_RECORDS"] < 96
            ]

            # CT_RECORDS value distribution
            record_counts = (
                x["CT_RECORDS"]
                .value_counts()
                .sort_index()
                .to_dict()
            )

            ft1_daily.append({
                "Date": pd.to_datetime(
                    date_string,
                    format="%Y%m%d"
                ),
                "Rows": len(x),
                "Complete": int(
                    (x["CT_RECORDS"] == 96).sum()
                ),
                "Incomplete": len(incomplete),
                "Incomplete_%": (
                    len(incomplete) / len(x) * 100
                    if len(x) > 0 else np.nan
                ),
                "CT_RECORDS_Distribution": str(record_counts),
                "Positive": int(
                    (x["QT_VOLUME_24HOUR"] > 0).sum()
                ),
                "Negative": int(
                    (x["QT_VOLUME_24HOUR"] < 0).sum()
                ),
                "Alarm_Detectors": int(
                    (x["CT_ALARM_24HOUR"] > 0).sum()
                )
            })

            del d, x, incomplete, row_keys, mask
            gc.collect()

    print("  ✅ Done")


# ------------------------------------------------------------
# 3. DAILY TABLE
# ------------------------------------------------------------

ft1_daily = (
    pd.DataFrame(ft1_daily)
    .sort_values("Date")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 4. DAYS WITH INCOMPLETENESS
# ------------------------------------------------------------

ft1_incomplete_days = ft1_daily[
    ft1_daily["Incomplete"] > 0
].copy()

print("\n========================================")
print("FT1 DAYS WITH INCOMPLETE RECORDS")
print("========================================")

print(
    f"Days with any incomplete FT1 detector: "
    f"{len(ft1_incomplete_days)}"
)

if len(ft1_incomplete_days) > 0:

    print(
        ft1_incomplete_days[
            [
                "Date",
                "Rows",
                "Complete",
                "Incomplete",
                "Incomplete_%",
                "CT_RECORDS_Distribution",
                "Positive",
                "Negative",
                "Alarm_Detectors"
            ]
        ].to_string(index=False)
    )


# ------------------------------------------------------------
# 5. HOW MANY DAYS ARE 100% COMPLETE?
# ------------------------------------------------------------

perfect_complete_days = (
    ft1_daily["Incomplete"] == 0
).sum()

print("\n========================================")
print("FT1 COMPLETENESS SUMMARY")
print("========================================")

print(
    f"Eligible days analyzed: "
    f"{len(ft1_daily)}"
)

print(
    f"100% complete FT1 days: "
    f"{perfect_complete_days}"
)

print(
    f"Days with incompleteness: "
    f"{len(ft1_incomplete_days)}"
)


# ------------------------------------------------------------
# 6. SAVE
# ------------------------------------------------------------

ft1_daily.to_csv(
    OUTPUT_FOLDER /
    "FT1_daily_quality_audit.csv",
    index=False
)

print("\n✅ FT1 systematic incompleteness audit complete.")

In [ ]:
# ============================================================
# NOTEBOOK 2 — REGIONAL SYSTEM-DISRUPTION DETECTION
# ============================================================

import pandas as pd
import numpy as np
import zipfile
import re
import gc

# ------------------------------------------------------------
# 1. ACTIVITY-QUALIFIED POOL
#    BEFORE completeness filtering
# ------------------------------------------------------------

regional_pool = stable_with_region[
    stable_with_region["Positive_Eligible_%"] >= 95
].copy()

regional_pool["Key"] = (
    regional_pool["NB_SCATS_SITE"].astype("int64") * 100
    + regional_pool["NB_DETECTOR"].astype("int64")
)

regional_keys = np.unique(
    regional_pool["Key"].to_numpy()
)

print("========================================")
print("REGIONAL DISRUPTION ANALYSIS")
print("========================================")

print(
    f"\nActivity-qualified detectors: "
    f"{len(regional_pool):,}"
)

print(
    f"Regions represented: "
    f"{regional_pool['NM_REGION'].nunique()}"
)

print(
    f"Global disruption days already excluded: "
    f"{len(major_date_set)}"
)


# ------------------------------------------------------------
# 2. DAILY REGION-LEVEL PROFILES
# ------------------------------------------------------------

USE_COLS_REGION = [
    "NB_SCATS_SITE",
    "NB_DETECTOR",
    "NM_REGION",
    "QT_VOLUME_24HOUR",
    "CT_RECORDS",
    "CT_ALARM_24HOUR"
]

regional_daily_records = []

print("\n========================================")
print("BUILDING REGION-DAY QUALITY TABLE")
print("========================================\n")

for zip_path in zip_files:

    print(f"Processing: {zip_path.name}")

    with zipfile.ZipFile(zip_path, "r") as z:

        csv_names = sorted([
            name for name in z.namelist()
            if name.lower().endswith(".csv")
        ])

        for csv_name in csv_names:

            match = re.search(
                r"VSDATA_(\d{8})\.csv",
                csv_name,
                re.IGNORECASE
            )

            if not match:
                continue

            date_string = match.group(1)

            # Do not classify global network failures
            # as regional failures
            if date_string in major_date_set:
                continue

            with z.open(csv_name) as f:

                df = pd.read_csv(
                    f,
                    usecols=USE_COLS_REGION,
                    dtype={
                        "NB_SCATS_SITE": "int32",
                        "NB_DETECTOR": "int16",
                        "NM_REGION": "string",
                        "QT_VOLUME_24HOUR": "int32",
                        "CT_RECORDS": "int16",
                        "CT_ALARM_24HOUR": "int16"
                    }
                )

            row_keys = (
                df["NB_SCATS_SITE"].astype("int64") * 100
                + df["NB_DETECTOR"].astype("int64")
            ).to_numpy()

            mask = np.isin(
                row_keys,
                regional_keys
            )

            x = df.loc[mask].copy()

            # Daily detector-quality flags
            x["Positive_Flag"] = (
                x["QT_VOLUME_24HOUR"] > 0
            ).astype("int8")

            x["Negative_Flag"] = (
                x["QT_VOLUME_24HOUR"] < 0
            ).astype("int8")

            x["Zero_Flag"] = (
                x["QT_VOLUME_24HOUR"] == 0
            ).astype("int8")

            x["Incomplete_Flag"] = (
                x["CT_RECORDS"] < 96
            ).astype("int8")

            x["Alarm_Flag"] = (
                x["CT_ALARM_24HOUR"] > 0
            ).astype("int8")

            day_region = (
                x.groupby(
                    "NM_REGION",
                    observed=True
                )
                .agg(
                    Rows=("NB_DETECTOR", "size"),
                    Positive=("Positive_Flag", "sum"),
                    Negative=("Negative_Flag", "sum"),
                    Zero=("Zero_Flag", "sum"),
                    Incomplete=("Incomplete_Flag", "sum"),
                    Alarm_Detectors=("Alarm_Flag", "sum")
                )
                .reset_index()
            )

            day_region["Date"] = pd.to_datetime(
                date_string,
                format="%Y%m%d"
            )

            regional_daily_records.append(
                day_region
            )

            del df
            del x
            del row_keys
            del mask
            del day_region
            gc.collect()

    print("  ✅ Done")


# ------------------------------------------------------------
# 3. COMBINE TABLE
# ------------------------------------------------------------

regional_daily = pd.concat(
    regional_daily_records,
    ignore_index=True
)

regional_daily = regional_daily.sort_values(
    ["Date", "NM_REGION"]
).reset_index(drop=True)


# ------------------------------------------------------------
# 4. PERCENTAGES
# ------------------------------------------------------------

for col in [
    "Positive",
    "Negative",
    "Zero",
    "Incomplete",
    "Alarm_Detectors"
]:

    regional_daily[f"{col}_%"] = (
        regional_daily[col]
        / regional_daily["Rows"]
        * 100
    )


# ------------------------------------------------------------
# 5. DEFINE MAJOR REGIONAL DISRUPTION
# ------------------------------------------------------------
#
# Strong threshold:
# >=80% of activity-qualified detectors affected
#
# We use this only for severe region-wide events.
# ------------------------------------------------------------

REGIONAL_MAJOR_THRESHOLD = 80.0

regional_daily[
    "Major_Regional_Disruption"
] = (
    (
        regional_daily["Negative_%"]
        >= REGIONAL_MAJOR_THRESHOLD
    )
    |
    (
        regional_daily["Incomplete_%"]
        >= REGIONAL_MAJOR_THRESHOLD
    )
)


# ------------------------------------------------------------
# 6. EXTRACT EVENTS
# ------------------------------------------------------------

regional_events = (
    regional_daily[
        regional_daily[
            "Major_Regional_Disruption"
        ]
    ]
    .copy()
    .sort_values(
        ["Date", "NM_REGION"]
    )
)


# ------------------------------------------------------------
# 7. VALIDATION
# ------------------------------------------------------------

expected_region_days = (
    234 *
    regional_pool["NM_REGION"].nunique()
)

print("\n========================================")
print("REGION-DAY TABLE VALIDATION")
print("========================================")

print(
    f"Region-day rows generated: "
    f"{len(regional_daily):,}"
)

print(
    f"Theoretical maximum "
    f"(234 × 40): "
    f"{expected_region_days:,}"
)

print(
    f"Unique dates: "
    f"{regional_daily['Date'].nunique()}"
)

print(
    f"Unique regions: "
    f"{regional_daily['NM_REGION'].nunique()}"
)


# ------------------------------------------------------------
# 8. MAJOR REGIONAL EVENTS
# ------------------------------------------------------------

print("\n========================================")
print("MAJOR REGIONAL DISRUPTION EVENTS")
print("========================================")

print(
    f"Number of region-day events: "
    f"{len(regional_events)}"
)

print(
    f"Regions affected: "
    f"{regional_events['NM_REGION'].nunique()}"
)

print(
    f"Unique dates involved: "
    f"{regional_events['Date'].nunique()}"
)

if len(regional_events) > 0:

    print("\n")

    print(
        regional_events[
            [
                "Date",
                "NM_REGION",
                "Rows",
                "Positive_%",
                "Negative_%",
                "Incomplete_%",
                "Alarm_Detectors_%"
            ]
        ]
        .round(3)
        .to_string(index=False)
    )


# ------------------------------------------------------------
# 9. EVENTS PER REGION
# ------------------------------------------------------------

print("\n========================================")
print("REGIONAL EVENTS BY REGION")
print("========================================")

if len(regional_events) > 0:

    regional_event_summary = (
        regional_events
        .groupby("NM_REGION")
        .size()
        .rename("Major_Events")
        .sort_values(
            ascending=False
        )
    )

    print(
        regional_event_summary.to_string()
    )

else:

    print("No major regional disruptions detected.")


# ------------------------------------------------------------
# 10. SAVE
# ------------------------------------------------------------

regional_daily.to_csv(
    OUTPUT_FOLDER /
    "regional_daily_quality.csv",
    index=False
)

regional_events.to_csv(
    OUTPUT_FOLDER /
    "major_regional_disruption_events.csv",
    index=False
)

print(
    "\n✅ Regional disruption detection complete."
)

In [ ]:
# ============================================================
# NOTEBOOK 2 — FINAL RECALCULATION
# GLOBAL + REGION-SPECIFIC DISRUPTION ADJUSTMENT
# ============================================================

import pandas as pd
import numpy as np
import zipfile
import re
import gc

# ------------------------------------------------------------
# 1. BUILD STABLE DETECTOR MASTER TABLE
# ------------------------------------------------------------

stable_adjusted = stable_with_region[
    [
        "NB_SCATS_SITE",
        "NB_DETECTOR",
        "NM_REGION"
    ]
].copy()

stable_adjusted["Key"] = (
    stable_adjusted["NB_SCATS_SITE"].astype("int64") * 100
    + stable_adjusted["NB_DETECTOR"].astype("int64")
)

stable_adjusted = (
    stable_adjusted
    .sort_values("Key")
    .reset_index(drop=True)
)

stable_keys = stable_adjusted["Key"].to_numpy()

N = len(stable_adjusted)

print("========================================")
print("FINAL GLOBAL + REGIONAL RECALCULATION")
print("========================================")

print(f"\nStable detectors: {N:,}")
print(
    f"Regions represented: "
    f"{stable_adjusted['NM_REGION'].nunique()}"
)


# ------------------------------------------------------------
# 2. GLOBAL DISRUPTION DATES
# ------------------------------------------------------------

global_dates = set(major_date_set)

print(
    f"Global disruption days excluded "
    f"for everyone: {len(global_dates)}"
)


# ------------------------------------------------------------
# 3. REGION-SPECIFIC DISRUPTION LOOKUP
# ------------------------------------------------------------

regional_event_lookup = {}

for _, row in regional_events.iterrows():

    date_string = pd.to_datetime(
        row["Date"]
    ).strftime("%Y%m%d")

    region = str(row["NM_REGION"])

    regional_event_lookup.setdefault(
        date_string,
        set()
    ).add(region)


print(
    f"Regional disruption events: "
    f"{len(regional_events)}"
)

print(
    f"Unique regional-event dates: "
    f"{len(regional_event_lookup)}"
)


# ------------------------------------------------------------
# 4. ACCUMULATORS
# ------------------------------------------------------------

eligible_days = np.zeros(
    N,
    dtype=np.int16
)

positive_days = np.zeros(
    N,
    dtype=np.int16
)

zero_days = np.zeros(
    N,
    dtype=np.int16
)

negative_days = np.zeros(
    N,
    dtype=np.int16
)

complete_days = np.zeros(
    N,
    dtype=np.int16
)

alarm_days = np.zeros(
    N,
    dtype=np.int16
)

positive_volume_sum = np.zeros(
    N,
    dtype=np.int64
)

match_errors = []

USE_COLS_FINAL = [
    "NB_SCATS_SITE",
    "NB_DETECTOR",
    "NM_REGION",
    "QT_VOLUME_24HOUR",
    "CT_RECORDS",
    "CT_ALARM_24HOUR"
]


# ------------------------------------------------------------
# 5. PROCESS ALL NON-GLOBAL DAYS
# ------------------------------------------------------------

print("\n========================================")
print("PROCESSING FINAL ELIGIBILITY PERIOD")
print("========================================\n")

processed_dates = 0

for zip_path in zip_files:

    print(f"Processing: {zip_path.name}")

    with zipfile.ZipFile(zip_path, "r") as z:

        csv_names = sorted([
            name for name in z.namelist()
            if name.lower().endswith(".csv")
        ])

        for csv_name in csv_names:

            match = re.search(
                r"VSDATA_(\d{8})\.csv",
                csv_name,
                re.IGNORECASE
            )

            if not match:
                continue

            date_string = match.group(1)

            # --------------------------------------------
            # GLOBAL disruption:
            # remove day for every detector
            # --------------------------------------------

            if date_string in global_dates:
                continue

            processed_dates += 1

            with z.open(csv_name) as f:

                df = pd.read_csv(
                    f,
                    usecols=USE_COLS_FINAL,
                    dtype={
                        "NB_SCATS_SITE": "int32",
                        "NB_DETECTOR": "int16",
                        "NM_REGION": "string",
                        "QT_VOLUME_24HOUR": "int32",
                        "CT_RECORDS": "int16",
                        "CT_ALARM_24HOUR": "int16"
                    }
                )

            row_keys = (
                df["NB_SCATS_SITE"].astype("int64") * 100
                + df["NB_DETECTOR"].astype("int64")
            ).to_numpy()

            positions = np.searchsorted(
                stable_keys,
                row_keys
            )

            valid = positions < N

            matched = np.zeros(
                len(row_keys),
                dtype=bool
            )

            matched[valid] = (
                stable_keys[positions[valid]]
                == row_keys[valid]
            )

            d = df.loc[
                matched,
                [
                    "NM_REGION",
                    "QT_VOLUME_24HOUR",
                    "CT_RECORDS",
                    "CT_ALARM_24HOUR"
                ]
            ].copy()

            detector_positions = positions[
                matched
            ]

            if len(d) != N:

                match_errors.append(
                    (date_string, len(d))
                )

            # --------------------------------------------
            # REGION-SPECIFIC disruption:
            # exclude only affected region(s)
            # --------------------------------------------

            affected_regions = (
                regional_event_lookup.get(
                    date_string,
                    set()
                )
            )

            if affected_regions:

                region_eligible = ~d[
                    "NM_REGION"
                ].isin(
                    affected_regions
                ).to_numpy()

            else:

                region_eligible = np.ones(
                    len(d),
                    dtype=bool
                )

            pos = detector_positions[
                region_eligible
            ]

            volume = (
                d.loc[
                    region_eligible,
                    "QT_VOLUME_24HOUR"
                ]
                .to_numpy()
            )

            records = (
                d.loc[
                    region_eligible,
                    "CT_RECORDS"
                ]
                .to_numpy()
            )

            alarms = (
                d.loc[
                    region_eligible,
                    "CT_ALARM_24HOUR"
                ]
                .to_numpy()
            )

            # --------------------------------------------
            # ACCUMULATE
            # --------------------------------------------

            eligible_days[pos] += 1

            positive_mask = volume > 0
            zero_mask = volume == 0
            negative_mask = volume < 0

            positive_days[pos] += (
                positive_mask.astype(np.int16)
            )

            zero_days[pos] += (
                zero_mask.astype(np.int16)
            )

            negative_days[pos] += (
                negative_mask.astype(np.int16)
            )

            complete_days[pos] += (
                (records == 96)
                .astype(np.int16)
            )

            alarm_days[pos] += (
                (alarms > 0)
                .astype(np.int16)
            )

            positive_volume_sum[pos] += np.where(
                positive_mask,
                volume,
                0
            ).astype(np.int64)

            del df
            del d
            del row_keys
            del positions
            del matched
            gc.collect()

    print("  ✅ Done")


# ------------------------------------------------------------
# 6. VALIDATION
# ------------------------------------------------------------

print("\n========================================")
print("FINAL RECALCULATION VALIDATION")
print("========================================")

print(
    f"Non-global dates processed: "
    f"{processed_dates}"
)

print(
    f"Expected: "
    f"{242 - len(global_dates)}"
)

print(
    f"Detector-count mismatch days: "
    f"{len(match_errors)}"
)

print(
    f"Minimum eligible days "
    f"for any detector: "
    f"{eligible_days.min()}"
)

print(
    f"Maximum eligible days "
    f"for any detector: "
    f"{eligible_days.max()}"
)


# ------------------------------------------------------------
# 7. CREATE FINAL ADJUSTED METRICS
# ------------------------------------------------------------

stable_adjusted[
    "Adjusted_Eligible_Days"
] = eligible_days

stable_adjusted[
    "Adjusted_Positive_Days"
] = positive_days

stable_adjusted[
    "Adjusted_Zero_Days"
] = zero_days

stable_adjusted[
    "Adjusted_Negative_Days"
] = negative_days

stable_adjusted[
    "Adjusted_Complete_Days"
] = complete_days

stable_adjusted[
    "Adjusted_Alarm_Days"
] = alarm_days


stable_adjusted[
    "Adjusted_Activity_%"
] = (
    positive_days
    / eligible_days
    * 100
)

stable_adjusted[
    "Adjusted_Completeness_%"
] = (
    complete_days
    / eligible_days
    * 100
)

stable_adjusted[
    "Adjusted_Alarm_%"
] = (
    alarm_days
    / eligible_days
    * 100
)


# Avoid divide-by-zero warning
stable_adjusted[
    "Adjusted_Mean_Positive_Daily_Volume"
] = 0.0

positive_nonzero = (
    positive_days > 0
)

stable_adjusted.loc[
    positive_nonzero,
    "Adjusted_Mean_Positive_Daily_Volume"
] = (
    positive_volume_sum[
        positive_nonzero
    ]
    / positive_days[
        positive_nonzero
    ]
)


# ------------------------------------------------------------
# 8. THRESHOLD RESULTS
# ------------------------------------------------------------

print("\n========================================")
print("ADJUSTED ACTIVITY RESULTS")
print("========================================")

for threshold in [
    90, 95, 98, 99
]:

    mask = (
        stable_adjusted[
            "Adjusted_Activity_%"
        ] >= threshold
    )

    print(
        f">= {threshold}% activity: "
        f"{mask.sum():,} detectors"
    )


# ------------------------------------------------------------
# 9. FINAL DETECTOR POOL
# ------------------------------------------------------------

FINAL_ACTIVITY = 95.0
FINAL_COMPLETENESS = 99.0

FINAL_detector_pool_adjusted = (
    stable_adjusted[
        (
            stable_adjusted[
                "Adjusted_Activity_%"
            ] >= FINAL_ACTIVITY
        )
        &
        (
            stable_adjusted[
                "Adjusted_Completeness_%"
            ] >= FINAL_COMPLETENESS
        )
    ]
    .copy()
)


# ------------------------------------------------------------
# 10. FINAL SUMMARY
# ------------------------------------------------------------

print("\n========================================")
print("FINAL ADJUSTED DETECTOR POOL")
print("========================================")

print(
    f"Final detectors: "
    f"{len(FINAL_detector_pool_adjusted):,}"
)

print(
    f"Final sites: "
    f"{FINAL_detector_pool_adjusted['NB_SCATS_SITE'].nunique():,}"
)

print(
    f"Final regions: "
    f"{FINAL_detector_pool_adjusted['NM_REGION'].nunique()}"
)

print(
    f"Median activity: "
    f"{FINAL_detector_pool_adjusted['Adjusted_Activity_%'].median():.2f}%"
)

print(
    f"Median completeness: "
    f"{FINAL_detector_pool_adjusted['Adjusted_Completeness_%'].median():.2f}%"
)

print(
    f"Median alarm rate: "
    f"{FINAL_detector_pool_adjusted['Adjusted_Alarm_%'].median():.2f}%"
)

print(
    f"Median positive daily volume: "
    f"{FINAL_detector_pool_adjusted['Adjusted_Mean_Positive_Daily_Volume'].median():,.1f}"
)


# FT1 check
ft1_final = FINAL_detector_pool_adjusted[
    FINAL_detector_pool_adjusted[
        "NM_REGION"
    ] == "FT1"
]

print("\n===== FT1 AFTER ADJUSTMENT =====")

print(
    f"FT1 detectors retained: "
    f"{len(ft1_final):,}"
)

print(
    f"FT1 sites represented: "
    f"{ft1_final['NB_SCATS_SITE'].nunique():,}"
)


# ------------------------------------------------------------
# 11. SAVE
# ------------------------------------------------------------

stable_adjusted.to_csv(
    OUTPUT_FOLDER /
    "stable_detectors_global_regional_adjusted.csv",
    index=False
)

FINAL_detector_pool_adjusted.to_csv(
    OUTPUT_FOLDER /
    "FINAL_detector_pool_ADJUSTED.csv",
    index=False
)

print(
    "\n✅ Global + regional recalculation complete."
)

In [ ]:
# ============================================================
# NOTEBOOK 2 — PREPARE MASTER MODELING POOL
# ============================================================

import pandas as pd
import numpy as np

modeling_pool = FINAL_detector_pool_adjusted.copy()

# ============================================================
# 1. UNIQUE DETECTOR ID
# ============================================================

modeling_pool["Detector_ID"] = (
    modeling_pool["NB_SCATS_SITE"].astype(str)
    + "_"
    + modeling_pool["NB_DETECTOR"].astype(str)
)


# ============================================================
# 2. TRAFFIC-VOLUME GROUP
# ============================================================

modeling_pool["Volume_Group"] = pd.cut(
    modeling_pool[
        "Adjusted_Mean_Positive_Daily_Volume"
    ],
    bins=[
        -np.inf,
        100,
        250,
        500,
        2000,
        5000,
        np.inf
    ],
    labels=[
        "<100",
        "100–249",
        "250–499",
        "500–1999",
        "2000–4999",
        "5000+"
    ],
    right=False
)


# ============================================================
# 3. REAL ALARM-EXPOSURE GROUP
# ============================================================

modeling_pool["Alarm_Exposure_Group"] = pd.cut(
    modeling_pool["Adjusted_Alarm_%"],
    bins=[
        -0.000001,
        0,
        1,
        2,
        5,
        np.inf
    ],
    labels=[
        "No alarms",
        ">0–1%",
        ">1–2%",
        ">2–5%",
        ">5%"
    ],
    include_lowest=True
)


# ============================================================
# 4. BASIC VALIDATION
# ============================================================

print("========================================")
print("MASTER MODELING POOL")
print("========================================")

print(
    f"Detectors: "
    f"{len(modeling_pool):,}"
)

print(
    f"Sites: "
    f"{modeling_pool['NB_SCATS_SITE'].nunique():,}"
)

print(
    f"Regions: "
    f"{modeling_pool['NM_REGION'].nunique()}"
)

print(
    f"Duplicate Detector_IDs: "
    f"{modeling_pool['Detector_ID'].duplicated().sum():,}"
)


# ============================================================
# 5. ADJUSTED ELIGIBLE-DAY DISTRIBUTION
# ============================================================

print("\n========================================")
print("ELIGIBLE DAYS AFTER SYSTEM ADJUSTMENT")
print("========================================")

print(
    modeling_pool[
        "Adjusted_Eligible_Days"
    ]
    .value_counts()
    .sort_index()
)


# ============================================================
# 6. VOLUME DISTRIBUTION
# ============================================================

print("\n========================================")
print("TRAFFIC-VOLUME GROUPS")
print("========================================")

volume_table = (
    modeling_pool["Volume_Group"]
    .value_counts(sort=False)
)

for group, count in volume_table.items():

    print(
        f"{str(group):<12}: "
        f"{count:>6,} "
        f"({count / len(modeling_pool) * 100:5.2f}%)"
    )


# ============================================================
# 7. ALARM EXPOSURE DISTRIBUTION
# ============================================================

print("\n========================================")
print("REAL ALARM-EXPOSURE GROUPS")
print("========================================")

alarm_table = (
    modeling_pool["Alarm_Exposure_Group"]
    .value_counts(sort=False)
)

for group, count in alarm_table.items():

    print(
        f"{str(group):<12}: "
        f"{count:>6,} "
        f"({count / len(modeling_pool) * 100:5.2f}%)"
    )


# ============================================================
# 8. REGION DISTRIBUTION
# ============================================================

print("\n========================================")
print("FINAL DETECTORS BY REGION")
print("========================================")

region_final = (
    modeling_pool
    .groupby("NM_REGION")
    .agg(
        Detectors=("Detector_ID", "size"),
        Sites=("NB_SCATS_SITE", "nunique")
    )
    .sort_values(
        "Detectors",
        ascending=False
    )
)

region_final["Detector_%"] = (
    region_final["Detectors"]
    / len(modeling_pool)
    * 100
)

print(
    region_final
    .round(2)
    .to_string()
)


# ============================================================
# 9. NUMBER OF ACTIVE DETECTORS PER SITE
# ============================================================

site_detector_counts = (
    modeling_pool
    .groupby("NB_SCATS_SITE")
    .size()
)

print("\n========================================")
print("DETECTORS PER SITE")
print("========================================")

print(
    site_detector_counts.describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)


# ============================================================
# 10. SAVE MASTER MODELING POOL
# ============================================================

MASTER_POOL_FILE = (
    OUTPUT_FOLDER /
    "MASTER_modeling_detector_pool.csv"
)

modeling_pool.to_csv(
    MASTER_POOL_FILE,
    index=False
)

region_final.to_csv(
    OUTPUT_FOLDER /
    "MASTER_modeling_pool_region_summary.csv"
)

print("\n========================================")
print("SAVED")
print("========================================")

print(MASTER_POOL_FILE)

print(
    "\n✅ Master modeling pool prepared successfully."
)

In [ ]:
# ============================================================
# NOTEBOOK 2 — REPRODUCIBLE STRATIFIED MODELING SAMPLE
# 800 DETECTORS FROM 800 UNIQUE SITES
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

SAMPLE_SIZE = 800
MIN_SITES_PER_REGION = 5
RANDOM_SEED = 2026

rng = np.random.default_rng(RANDOM_SEED)

pool = modeling_pool.copy()


# ============================================================
# 1. CREATE COMBINED QUALITY / VOLUME STRATUM
# ============================================================

pool["Sampling_Stratum"] = (
    pool["Volume_Group"].astype(str)
    + " | "
    + pool["Alarm_Exposure_Group"].astype(str)
)


# ============================================================
# 2. ONE CANDIDATE DETECTOR PER SITE
# ============================================================
#
# Randomly choose one detector from each site first.
# This prevents the final sample from containing multiple
# detectors from the same intersection/site.
# ============================================================

site_candidates = (
    pool
    .sample(
        frac=1,
        random_state=RANDOM_SEED
    )
    .drop_duplicates(
        subset=["NB_SCATS_SITE"],
        keep="first"
    )
    .reset_index(drop=True)
)

print("========================================")
print("SITE-LEVEL CANDIDATE POOL")
print("========================================")

print(
    f"Candidate detectors: "
    f"{len(site_candidates):,}"
)

print(
    f"Unique sites: "
    f"{site_candidates['NB_SCATS_SITE'].nunique():,}"
)

print(
    f"Regions: "
    f"{site_candidates['NM_REGION'].nunique()}"
)


# ============================================================
# 3. QUOTA ALLOCATION FUNCTION
# ============================================================

def allocate_quotas(
    counts,
    target,
    min_each=0
):
    """
    Deterministic proportional quota allocation
    with optional minimum per group.
    """

    counts = counts.astype(int)

    quota = pd.Series(
        0,
        index=counts.index,
        dtype=int
    )

    # Minimum allocation
    for idx in counts.index:
        quota.loc[idx] = min(
            min_each,
            counts.loc[idx]
        )

    if quota.sum() > target:
        raise ValueError(
            "Minimum allocation exceeds target."
        )

    ideal = (
        counts / counts.sum() * target
    )

    # Add one unit at a time until target reached
    while quota.sum() < target:

        capacity = counts - quota

        eligible = capacity[
            capacity > 0
        ].index

        if len(eligible) == 0:
            break

        priority = (
            ideal.loc[eligible]
            - quota.loc[eligible]
        )

        chosen = priority.idxmax()

        quota.loc[chosen] += 1

    return quota


# ============================================================
# 4. REGION QUOTAS
# ============================================================

region_available = (
    site_candidates["NM_REGION"]
    .value_counts()
    .sort_index()
)

region_quota = allocate_quotas(
    region_available,
    SAMPLE_SIZE,
    min_each=MIN_SITES_PER_REGION
)

print("\n========================================")
print("REGION SAMPLE QUOTAS")
print("========================================")

region_quota_table = pd.DataFrame({
    "Available_Sites": region_available,
    "Sample_Target": region_quota
})

region_quota_table["Sampling_%"] = (
    region_quota_table["Sample_Target"]
    / region_quota_table["Available_Sites"]
    * 100
)

print(
    region_quota_table
    .sort_values(
        "Sample_Target",
        ascending=False
    )
    .round(2)
    .to_string()
)

print(
    f"\nTotal allocated sample: "
    f"{region_quota.sum():,}"
)


# ============================================================
# 5. SAMPLE WITHIN EACH REGION
#    STRATIFIED BY VOLUME + ALARM EXPOSURE
# ============================================================

selected_parts = []

for region in region_quota.index:

    target_n = int(
        region_quota.loc[region]
    )

    region_df = site_candidates[
        site_candidates["NM_REGION"]
        == region
    ].copy()

    stratum_counts = (
        region_df[
            "Sampling_Stratum"
        ]
        .value_counts()
    )

    # If region quota is large enough,
    # try to preserve every non-empty stratum.
    if target_n >= len(stratum_counts):
        stratum_min = 1
    else:
        stratum_min = 0

    stratum_quota = allocate_quotas(
        stratum_counts,
        target_n,
        min_each=stratum_min
    )

    for stratum, n_select in stratum_quota.items():

        if n_select <= 0:
            continue

        candidates = region_df[
            region_df["Sampling_Stratum"]
            == stratum
        ]

        selected = candidates.sample(
            n=int(n_select),
            random_state=(
                RANDOM_SEED
                + sum(ord(c) for c in str(region))
                + sum(ord(c) for c in str(stratum))
            )
        )

        selected_parts.append(
            selected
        )


# ============================================================
# 6. FINAL MODELING SAMPLE
# ============================================================

modeling_sample = pd.concat(
    selected_parts,
    ignore_index=True
)

modeling_sample = (
    modeling_sample
    .sort_values(
        [
            "NM_REGION",
            "NB_SCATS_SITE",
            "NB_DETECTOR"
        ]
    )
    .reset_index(drop=True)
)


# ============================================================
# 7. HARD VALIDATION
# ============================================================

print("\n========================================")
print("MODELING SAMPLE VALIDATION")
print("========================================")

print(
    f"Selected detectors: "
    f"{len(modeling_sample):,}"
)

print(
    f"Unique detector IDs: "
    f"{modeling_sample['Detector_ID'].nunique():,}"
)

print(
    f"Unique sites: "
    f"{modeling_sample['NB_SCATS_SITE'].nunique():,}"
)

print(
    f"Regions represented: "
    f"{modeling_sample['NM_REGION'].nunique()}"
)

print(
    f"Duplicate sites: "
    f"{modeling_sample['NB_SCATS_SITE'].duplicated().sum():,}"
)

print(
    f"Duplicate detectors: "
    f"{modeling_sample['Detector_ID'].duplicated().sum():,}"
)


# ============================================================
# 8. SAMPLE VOLUME DISTRIBUTION
# ============================================================

print("\n========================================")
print("SAMPLE — VOLUME GROUPS")
print("========================================")

sample_volume = (
    modeling_sample["Volume_Group"]
    .value_counts(sort=False)
)

for group, count in sample_volume.items():

    print(
        f"{str(group):<12}: "
        f"{count:>4,} "
        f"({count / len(modeling_sample) * 100:5.2f}%)"
    )


# ============================================================
# 9. SAMPLE ALARM DISTRIBUTION
# ============================================================

print("\n========================================")
print("SAMPLE — ALARM EXPOSURE")
print("========================================")

sample_alarm = (
    modeling_sample[
        "Alarm_Exposure_Group"
    ]
    .value_counts(sort=False)
)

for group, count in sample_alarm.items():

    print(
        f"{str(group):<12}: "
        f"{count:>4,} "
        f"({count / len(modeling_sample) * 100:5.2f}%)"
    )


# ============================================================
# 10. SAMPLE BY REGION
# ============================================================

print("\n========================================")
print("SAMPLE — REGION DISTRIBUTION")
print("========================================")

sample_region = (
    modeling_sample
    .groupby("NM_REGION")
    .agg(
        Detectors=("Detector_ID", "size"),
        Sites=("NB_SCATS_SITE", "nunique")
    )
    .sort_values(
        "Detectors",
        ascending=False
    )
)

print(
    sample_region.to_string()
)


# ============================================================
# 11. ESTIMATED DATA SIZE
# ============================================================

detector_days = (
    modeling_sample[
        "Adjusted_Eligible_Days"
    ].sum()
)

estimated_intervals = (
    detector_days * 96
)

print("\n========================================")
print("EXPECTED MODELING DATA SIZE")
print("========================================")

print(
    f"Eligible detector-days: "
    f"{detector_days:,}"
)

print(
    f"Approximate 15-min observations: "
    f"{estimated_intervals:,}"
)


# ============================================================
# 12. SAVE
# ============================================================

SAMPLE_FILE = (
    OUTPUT_FOLDER /
    "MODELING_SAMPLE_800_DETECTORS.csv"
)

modeling_sample.to_csv(
    SAMPLE_FILE,
    index=False
)

region_quota_table.to_csv(
    OUTPUT_FOLDER /
    "MODELING_SAMPLE_region_quotas.csv"
)

print("\n========================================")
print("SAVED")
print("========================================")

print(SAMPLE_FILE)

print(
    "\n✅ Reproducible stratified modeling sample created."
)

In [ ]:
# ============================================================
# NOTEBOOK 2 — MODELING SAMPLE REPRESENTATIVENESS AUDIT
# ============================================================

import pandas as pd
import numpy as np

print("========================================")
print("MODELING SAMPLE REPRESENTATIVENESS AUDIT")
print("========================================")


# ============================================================
# HELPER FUNCTION
# ============================================================

def compare_distribution(reference, sample, column, order=None):

    ref_pct = (
        reference[column]
        .value_counts(normalize=True, sort=False)
        * 100
    )

    sample_pct = (
        sample[column]
        .value_counts(normalize=True, sort=False)
        * 100
    )

    comparison = pd.concat(
        [
            ref_pct.rename("Reference_%"),
            sample_pct.rename("Sample_%")
        ],
        axis=1
    ).fillna(0)

    if order is not None:
        comparison = comparison.reindex(order).fillna(0)

    comparison["Difference_pp"] = (
        comparison["Sample_%"]
        - comparison["Reference_%"]
    )

    comparison["Abs_Difference_pp"] = (
        comparison["Difference_pp"].abs()
    )

    return comparison


# ============================================================
# 1. VOLUME DISTRIBUTION
# Reference = one candidate detector per unique site
# ============================================================

volume_order = [
    "<100",
    "100–249",
    "250–499",
    "500–1999",
    "2000–4999",
    "5000+"
]

volume_comparison = compare_distribution(
    site_candidates,
    modeling_sample,
    "Volume_Group",
    volume_order
)

print("\n========================================")
print("VOLUME DISTRIBUTION")
print("Reference = 4,302 unique-site candidates")
print("========================================")

print(
    volume_comparison
    .round(2)
    .to_string()
)

print(
    "\nMaximum absolute difference: "
    f"{volume_comparison['Abs_Difference_pp'].max():.2f} percentage points"
)


# ============================================================
# 2. ALARM EXPOSURE DISTRIBUTION
# ============================================================

alarm_order = [
    "No alarms",
    ">0–1%",
    ">1–2%",
    ">2–5%",
    ">5%"
]

alarm_comparison = compare_distribution(
    site_candidates,
    modeling_sample,
    "Alarm_Exposure_Group",
    alarm_order
)

print("\n========================================")
print("ALARM-EXPOSURE DISTRIBUTION")
print("========================================")

print(
    alarm_comparison
    .round(2)
    .to_string()
)

print(
    "\nMaximum absolute difference: "
    f"{alarm_comparison['Abs_Difference_pp'].max():.2f} percentage points"
)


# ============================================================
# 3. REGION DISTRIBUTION
# ============================================================

region_comparison = compare_distribution(
    site_candidates,
    modeling_sample,
    "NM_REGION"
)

region_comparison = region_comparison.sort_values(
    "Abs_Difference_pp",
    ascending=False
)

print("\n========================================")
print("REGION DISTRIBUTION — TOP DIFFERENCES")
print("========================================")

print(
    region_comparison
    .head(15)
    .round(2)
    .to_string()
)


# ============================================================
# 4. STRATUM-LEVEL SAMPLING WEIGHTS
# ============================================================

reference_strata = (
    site_candidates
    .groupby(
        [
            "NM_REGION",
            "Sampling_Stratum"
        ],
        observed=True
    )
    .size()
    .rename("Available")
    .reset_index()
)

sample_strata = (
    modeling_sample
    .groupby(
        [
            "NM_REGION",
            "Sampling_Stratum"
        ],
        observed=True
    )
    .size()
    .rename("Selected")
    .reset_index()
)

stratum_weights = reference_strata.merge(
    sample_strata,
    on=[
        "NM_REGION",
        "Sampling_Stratum"
    ],
    how="left"
)

stratum_weights["Selected"] = (
    stratum_weights["Selected"]
    .fillna(0)
    .astype(int)
)

stratum_weights["Sampling_Fraction"] = np.where(
    stratum_weights["Available"] > 0,
    stratum_weights["Selected"]
    / stratum_weights["Available"],
    np.nan
)

stratum_weights["Sampling_Weight"] = np.where(
    stratum_weights["Selected"] > 0,
    stratum_weights["Available"]
    / stratum_weights["Selected"],
    np.nan
)


# ============================================================
# 5. ATTACH WEIGHTS TO MODELING SAMPLE
# ============================================================

modeling_sample_weighted = modeling_sample.merge(
    stratum_weights[
        [
            "NM_REGION",
            "Sampling_Stratum",
            "Available",
            "Selected",
            "Sampling_Fraction",
            "Sampling_Weight"
        ]
    ],
    on=[
        "NM_REGION",
        "Sampling_Stratum"
    ],
    how="left",
    validate="many_to_one"
)


# ============================================================
# 6. WEIGHT VALIDATION
# ============================================================

print("\n========================================")
print("SAMPLING-WEIGHT VALIDATION")
print("========================================")

print(
    f"Sample rows: "
    f"{len(modeling_sample_weighted):,}"
)

print(
    f"Missing sampling weights: "
    f"{modeling_sample_weighted['Sampling_Weight'].isna().sum():,}"
)

print(
    f"Minimum sampling weight: "
    f"{modeling_sample_weighted['Sampling_Weight'].min():.2f}"
)

print(
    f"Median sampling weight: "
    f"{modeling_sample_weighted['Sampling_Weight'].median():.2f}"
)

print(
    f"Maximum sampling weight: "
    f"{modeling_sample_weighted['Sampling_Weight'].max():.2f}"
)


# ============================================================
# 7. SAVE
# ============================================================

modeling_sample_weighted.to_csv(
    OUTPUT_FOLDER /
    "MODELING_SAMPLE_800_DETECTORS_WEIGHTED.csv",
    index=False
)

volume_comparison.to_csv(
    OUTPUT_FOLDER /
    "MODELING_SAMPLE_volume_representativeness.csv"
)

alarm_comparison.to_csv(
    OUTPUT_FOLDER /
    "MODELING_SAMPLE_alarm_representativeness.csv"
)

stratum_weights.to_csv(
    OUTPUT_FOLDER /
    "MODELING_SAMPLE_stratum_weights.csv",
    index=False
)

print(
    "\n✅ Representativeness audit complete."
)

In [ ]:
# ============================================================
# NOTEBOOK 2 — EXTRACT WIDE MODELING DATASET
# 800 DETECTORS × 242 DAYS × V00–V95
# ============================================================

import pandas as pd
import numpy as np
import zipfile
import re
import gc
from pathlib import Path

# ============================================================
# 1. SETTINGS
# ============================================================

V_COLS = [f"V{i:02d}" for i in range(96)]

READ_COLS = [
    "NB_SCATS_SITE",
    "NB_DETECTOR",
    "NM_REGION",
    "QT_VOLUME_24HOUR",
    "CT_RECORDS",
    "CT_ALARM_24HOUR"
] + V_COLS

WIDE_OUTPUT_FOLDER = (
    OUTPUT_FOLDER /
    "Modeling_Wide_Monthly"
)

WIDE_OUTPUT_FOLDER.mkdir(
    exist_ok=True
)

EXPECTED_DETECTORS = 800
EXPECTED_DAYS = 242
EXPECTED_ROWS = (
    EXPECTED_DETECTORS *
    EXPECTED_DAYS
)


# ============================================================
# 2. SAMPLE DETECTOR KEYS
# ============================================================

sample_detector_keys = np.unique(
    (
        modeling_sample_weighted[
            "NB_SCATS_SITE"
        ].astype("int64") * 100
        +
        modeling_sample_weighted[
            "NB_DETECTOR"
        ].astype("int64")
    ).to_numpy()
)

print("========================================")
print("WIDE DATASET EXTRACTION")
print("========================================")

print(
    f"\nSample detector keys: "
    f"{len(sample_detector_keys):,}"
)


# ============================================================
# 3. SAMPLE METADATA
# ============================================================

sample_metadata = (
    modeling_sample_weighted[
        [
            "Detector_ID",
            "Volume_Group",
            "Alarm_Exposure_Group",
            "Sampling_Stratum",
            "Sampling_Weight",
            "Adjusted_Eligible_Days",
            "Adjusted_Activity_%",
            "Adjusted_Completeness_%",
            "Adjusted_Alarm_%",
            "Adjusted_Mean_Positive_Daily_Volume"
        ]
    ]
    .copy()
)

# Convert categories to string for clean parquet saving
for col in [
    "Volume_Group",
    "Alarm_Exposure_Group"
]:
    sample_metadata[col] = (
        sample_metadata[col]
        .astype(str)
    )


# ============================================================
# 4. NETWORK DAY-CLASS LOOKUP
# ============================================================

day_class_lookup = {}

for _, row in dq2.iterrows():

    date_key = pd.to_datetime(
        row["Date"]
    ).strftime("%Y%m%d")

    day_class_lookup[date_key] = (
        row["Day_Class"]
    )


# ============================================================
# 5. SORT ZIP FILES CHRONOLOGICALLY
# ============================================================

def get_zip_first_date(zip_path):

    dates = []

    with zipfile.ZipFile(
        zip_path,
        "r"
    ) as z:

        for name in z.namelist():

            match = re.search(
                r"VSDATA_(\d{8})\.csv",
                name,
                re.IGNORECASE
            )

            if match:
                dates.append(
                    match.group(1)
                )

    return min(dates)


zip_files_chrono = sorted(
    zip_files,
    key=get_zip_first_date
)

print("\nZIP processing order:")

for zf in zip_files_chrono:
    print(
        f"  {get_zip_first_date(zf)} "
        f"→ {zf.name}"
    )


# ============================================================
# 6. EXTRACTION
# ============================================================

all_months = []

date_count_errors = []

print("\n========================================")
print("EXTRACTING SAMPLE DATA")
print("========================================\n")

for zip_path in zip_files_chrono:

    month_parts = []

    with zipfile.ZipFile(
        zip_path,
        "r"
    ) as z:

        csv_records = []

        for name in z.namelist():

            match = re.search(
                r"VSDATA_(\d{8})\.csv",
                name,
                re.IGNORECASE
            )

            if match:

                csv_records.append(
                    (
                        match.group(1),
                        name
                    )
                )

        # Explicit chronological order
        csv_records = sorted(
            csv_records,
            key=lambda x: x[0]
        )

        for date_string, csv_name in csv_records:

            with z.open(csv_name) as f:

                df = pd.read_csv(
                    f,
                    usecols=READ_COLS,
                    dtype={
                        "NB_SCATS_SITE": "int32",
                        "NB_DETECTOR": "int16",
                        "NM_REGION": "string",
                        "QT_VOLUME_24HOUR": "int32",
                        "CT_RECORDS": "int16",
                        "CT_ALARM_24HOUR": "int16",
                        **{
                            col: "int32"
                            for col in V_COLS
                        }
                    }
                )

            # --------------------------------------------
            # FILTER 800 SELECTED DETECTORS
            # --------------------------------------------

            row_keys = (
                df[
                    "NB_SCATS_SITE"
                ].astype("int64") * 100
                +
                df[
                    "NB_DETECTOR"
                ].astype("int64")
            ).to_numpy()

            mask = np.isin(
                row_keys,
                sample_detector_keys
            )

            d = df.loc[
                mask
            ].copy()

            # --------------------------------------------
            # VALIDATE 800 ROWS PER DAY
            # --------------------------------------------

            if len(d) != EXPECTED_DETECTORS:

                date_count_errors.append(
                    (
                        date_string,
                        len(d)
                    )
                )

            # --------------------------------------------
            # DATE / DETECTOR ID
            # --------------------------------------------

            d["Date"] = pd.to_datetime(
                date_string,
                format="%Y%m%d"
            )

            d["Detector_ID"] = (
                d["NB_SCATS_SITE"].astype(str)
                + "_"
                + d["NB_DETECTOR"].astype(str)
            )

            # --------------------------------------------
            # INTERVAL-LEVEL QUALITY COUNTS
            # --------------------------------------------

            interval_array = (
                d[V_COLS]
                .to_numpy()
            )

            d[
                "Negative_Interval_Count"
            ] = (
                interval_array < 0
            ).sum(axis=1).astype("int16")

            d[
                "Zero_Interval_Count"
            ] = (
                interval_array == 0
            ).sum(axis=1).astype("int16")

            d[
                "Positive_Interval_Count"
            ] = (
                interval_array > 0
            ).sum(axis=1).astype("int16")

            d[
                "Valid_Interval_Count"
            ] = (
                interval_array >= 0
            ).sum(axis=1).astype("int16")

            # --------------------------------------------
            # LOCAL QUALITY FLAGS
            # --------------------------------------------

            d["Has_Negative_Interval"] = (
                d[
                    "Negative_Interval_Count"
                ] > 0
            )

            d["Has_Alarm"] = (
                d["CT_ALARM_24HOUR"] > 0
            )

            d["Is_Incomplete"] = (
                d["CT_RECORDS"] < 96
            )

            d["Positive_Daily_Volume"] = (
                d["QT_VOLUME_24HOUR"] > 0
            )

            # --------------------------------------------
            # SYSTEM-LEVEL FLAGS
            # --------------------------------------------

            is_global = (
                date_string
                in global_dates
            )

            d[
                "Is_Global_Disruption"
            ] = is_global

            affected_regions = (
                regional_event_lookup.get(
                    date_string,
                    set()
                )
            )

            d[
                "Is_Regional_Disruption"
            ] = (
                d["NM_REGION"]
                .astype(str)
                .isin(
                    affected_regions
                )
            )

            d[
                "System_Eligible"
            ] = ~(
                d[
                    "Is_Global_Disruption"
                ]
                |
                d[
                    "Is_Regional_Disruption"
                ]
            )

            d[
                "Network_Day_Class"
            ] = day_class_lookup.get(
                date_string,
                "Unknown"
            )

            month_parts.append(d)

            del df
            del d
            del row_keys
            del mask
            del interval_array

            gc.collect()


    # ========================================================
    # MONTH TABLE
    # ========================================================

    month_df = pd.concat(
        month_parts,
        ignore_index=True
    )

    # Attach sampling / detector metadata
    month_df = month_df.merge(
        sample_metadata,
        on="Detector_ID",
        how="left",
        validate="many_to_one"
    )

    month_df = (
        month_df
        .sort_values(
            [
                "Date",
                "NB_SCATS_SITE",
                "NB_DETECTOR"
            ]
        )
        .reset_index(drop=True)
    )

    month_start = (
        month_df["Date"]
        .min()
        .strftime("%Y-%m")
    )

    monthly_file = (
        WIDE_OUTPUT_FOLDER /
        f"MODELING_WIDE_{month_start}.parquet"
    )

    month_df.to_parquet(
        monthly_file,
        index=False
    )

    print(
        f"{month_start}: "
        f"{len(month_df):,} rows "
        f"→ saved"
    )

    all_months.append(
        month_df
    )

    del month_parts
    gc.collect()


# ============================================================
# 7. COMBINE 8 MONTHS
# ============================================================

wide_modeling = pd.concat(
    all_months,
    ignore_index=True
)

wide_modeling = (
    wide_modeling
    .sort_values(
        [
            "Date",
            "NB_SCATS_SITE",
            "NB_DETECTOR"
        ]
    )
    .reset_index(drop=True)
)


# ============================================================
# 8. HARD VALIDATION
# ============================================================

print("\n========================================")
print("WIDE MODELING DATASET VALIDATION")
print("========================================")

print(
    f"Total rows: "
    f"{len(wide_modeling):,}"
)

print(
    f"Expected rows: "
    f"{EXPECTED_ROWS:,}"
)

print(
    f"Unique dates: "
    f"{wide_modeling['Date'].nunique()}"
)

print(
    f"Unique detectors: "
    f"{wide_modeling['Detector_ID'].nunique():,}"
)

print(
    f"Unique sites: "
    f"{wide_modeling['NB_SCATS_SITE'].nunique():,}"
)

print(
    f"Regions: "
    f"{wide_modeling['NM_REGION'].nunique()}"
)

duplicate_keys = (
    wide_modeling
    .duplicated(
        subset=[
            "Date",
            "NB_SCATS_SITE",
            "NB_DETECTOR"
        ]
    )
    .sum()
)

print(
    f"Duplicate detector-day rows: "
    f"{duplicate_keys:,}"
)

rows_per_day = (
    wide_modeling
    .groupby("Date")
    .size()
)

print(
    f"Minimum rows/day: "
    f"{rows_per_day.min():,}"
)

print(
    f"Maximum rows/day: "
    f"{rows_per_day.max():,}"
)

rows_per_detector = (
    wide_modeling
    .groupby("Detector_ID")
    .size()
)

print(
    f"Minimum days/detector: "
    f"{rows_per_detector.min()}"
)

print(
    f"Maximum days/detector: "
    f"{rows_per_detector.max()}"
)

print(
    f"Daily count errors during extraction: "
    f"{len(date_count_errors)}"
)


# ============================================================
# 9. SYSTEM-LEVEL EVENT COUNTS
# ============================================================

print("\n========================================")
print("SYSTEM EVENT FLAGS")
print("========================================")

print(
    f"Global disruption detector-days: "
    f"{wide_modeling['Is_Global_Disruption'].sum():,}"
)

print(
    f"Regional disruption detector-days: "
    f"{wide_modeling['Is_Regional_Disruption'].sum():,}"
)

print(
    f"System-eligible detector-days: "
    f"{wide_modeling['System_Eligible'].sum():,}"
)


# ============================================================
# 10. LOCAL DATA-QUALITY COUNTS
# ============================================================

print("\n========================================")
print("LOCAL QUALITY SUMMARY")
print("========================================")

print(
    f"Detector-days with alarms: "
    f"{wide_modeling['Has_Alarm'].sum():,}"
)

print(
    f"Detector-days with negative intervals: "
    f"{wide_modeling['Has_Negative_Interval'].sum():,}"
)

print(
    f"Incomplete detector-days: "
    f"{wide_modeling['Is_Incomplete'].sum():,}"
)

print(
    f"Positive daily-volume rows: "
    f"{wide_modeling['Positive_Daily_Volume'].sum():,}"
)


# ============================================================
# 11. SAVE COMPLETE MASTER WIDE DATASET
# ============================================================

MASTER_WIDE_FILE = (
    OUTPUT_FOLDER /
    "MODELING_WIDE_800_DETECTORS_242_DAYS.parquet"
)

wide_modeling.to_parquet(
    MASTER_WIDE_FILE,
    index=False
)

print("\n========================================")
print("MASTER WIDE DATASET SAVED")
print("========================================")

print(MASTER_WIDE_FILE)

print(
    "\n✅ Wide modeling dataset extraction complete."
)

In [ ]:
# ============================================================
# NOTEBOOK 2 — INTERVAL-LEVEL QUALITY AUDIT
# V00–V95 FOR THE 800-DETECTOR MODELING SAMPLE
# ============================================================

import pandas as pd
import numpy as np

print("========================================")
print("INTERVAL-LEVEL QUALITY AUDIT")
print("========================================")

V_COLS = [f"V{i:02d}" for i in range(96)]

# Use existing wide dataset
w = wide_modeling.copy()

interval_array = w[V_COLS].to_numpy(copy=False)

TOTAL_ROWS = len(w)
TOTAL_INTERVALS = TOTAL_ROWS * 96

print(f"\nDetector-day rows: {TOTAL_ROWS:,}")
print(f"Total 15-min observations: {TOTAL_INTERVALS:,}")


# ============================================================
# 1. OVERALL INTERVAL VALUES
# ============================================================

total_negative = int(
    (interval_array < 0).sum()
)

total_zero = int(
    (interval_array == 0).sum()
)

total_positive = int(
    (interval_array > 0).sum()
)

print("\n========================================")
print("OVERALL INTERVAL VALUE SUMMARY")
print("========================================")

print(
    f"Negative intervals: {total_negative:,} "
    f"({total_negative / TOTAL_INTERVALS * 100:.3f}%)"
)

print(
    f"Zero intervals: {total_zero:,} "
    f"({total_zero / TOTAL_INTERVALS * 100:.3f}%)"
)

print(
    f"Positive intervals: {total_positive:,} "
    f"({total_positive / TOTAL_INTERVALS * 100:.3f}%)"
)

print(
    f"Maximum interval volume: "
    f"{interval_array.max():,}"
)


# ============================================================
# 2. NEGATIVE VALUE CODES
# ============================================================

negative_values = interval_array[
    interval_array < 0
]

print("\n========================================")
print("NEGATIVE VALUE CODES")
print("========================================")

if len(negative_values) > 0:

    values, counts = np.unique(
        negative_values,
        return_counts=True
    )

    for value, count in zip(values, counts):
        print(
            f"{value}: {count:,}"
        )

else:
    print("No negative interval values.")


# ============================================================
# 3. SYSTEM-ELIGIBLE INTERVAL SUMMARY
# ============================================================

eligible_mask = (
    w["System_Eligible"]
    .to_numpy(dtype=bool)
)

eligible_intervals = interval_array[
    eligible_mask
]

eligible_total = (
    eligible_intervals.size
)

eligible_negative = int(
    (eligible_intervals < 0).sum()
)

eligible_zero = int(
    (eligible_intervals == 0).sum()
)

eligible_positive = int(
    (eligible_intervals > 0).sum()
)

print("\n========================================")
print("SYSTEM-ELIGIBLE INTERVALS")
print("========================================")

print(
    f"Eligible detector-days: "
    f"{eligible_mask.sum():,}"
)

print(
    f"Eligible intervals: "
    f"{eligible_total:,}"
)

print(
    f"Negative intervals: "
    f"{eligible_negative:,} "
    f"({eligible_negative / eligible_total * 100:.3f}%)"
)

print(
    f"Zero intervals: "
    f"{eligible_zero:,} "
    f"({eligible_zero / eligible_total * 100:.3f}%)"
)

print(
    f"Positive intervals: "
    f"{eligible_positive:,} "
    f"({eligible_positive / eligible_total * 100:.3f}%)"
)


# ============================================================
# 4. LOCAL QUALITY FLAG COMBINATIONS
#    SYSTEM-ELIGIBLE ROWS ONLY
# ============================================================

eligible_rows = w[
    w["System_Eligible"]
].copy()

quality_combo = (
    eligible_rows
    .groupby(
        [
            "Has_Alarm",
            "Has_Negative_Interval",
            "Is_Incomplete"
        ],
        dropna=False
    )
    .size()
    .rename("Detector_Days")
    .reset_index()
)

quality_combo["Percent"] = (
    quality_combo["Detector_Days"]
    / len(eligible_rows)
    * 100
)

quality_combo = quality_combo.sort_values(
    "Detector_Days",
    ascending=False
)

print("\n========================================")
print("LOCAL QUALITY FLAG COMBINATIONS")
print("SYSTEM-ELIGIBLE DAYS ONLY")
print("========================================")

print(
    quality_combo
    .round(3)
    .to_string(index=False)
)


# ============================================================
# 5. ALARM vs NEGATIVE INTERVAL RELATIONSHIP
# ============================================================

print("\n========================================")
print("ALARM vs NEGATIVE INTERVALS")
print("========================================")

alarm_negative_table = pd.crosstab(
    eligible_rows["Has_Alarm"],
    eligible_rows["Has_Negative_Interval"],
    margins=True
)

print(alarm_negative_table)


# ============================================================
# 6. CT_ALARM_24HOUR vs NUMBER OF -1 INTERVALS
# ============================================================

eligible_rows[
    "Alarm_minus_Negative_Count"
] = (
    eligible_rows["CT_ALARM_24HOUR"]
    - eligible_rows["Negative_Interval_Count"]
)

exact_alarm_match = (
    eligible_rows[
        "Alarm_minus_Negative_Count"
    ] == 0
).sum()

print("\n========================================")
print("ALARM COUNT vs NEGATIVE INTERVAL COUNT")
print("========================================")

print(
    f"Exact matches: "
    f"{exact_alarm_match:,} / {len(eligible_rows):,} "
    f"({exact_alarm_match / len(eligible_rows) * 100:.3f}%)"
)

print("\nMost common differences:")

print(
    eligible_rows[
        "Alarm_minus_Negative_Count"
    ]
    .value_counts()
    .head(15)
    .to_string()
)


# ============================================================
# 7. DAILY TOTAL CONSISTENCY
# ============================================================

w["Interval_Sum"] = (
    interval_array
    .astype("int64")
    .sum(axis=1)
)

w["Daily_Total_Match"] = (
    w["Interval_Sum"]
    == w["QT_VOLUME_24HOUR"]
)

print("\n========================================")
print("DAILY TOTAL CONSISTENCY")
print("========================================")

print(
    f"All rows matching QT_VOLUME_24HOUR: "
    f"{w['Daily_Total_Match'].sum():,} / {len(w):,} "
    f"({w['Daily_Total_Match'].mean() * 100:.3f}%)"
)

eligible_match = w.loc[
    w["System_Eligible"],
    "Daily_Total_Match"
]

print(
    f"System-eligible rows matching: "
    f"{eligible_match.sum():,} / {len(eligible_match):,} "
    f"({eligible_match.mean() * 100:.3f}%)"
)


# ============================================================
# 8. STRUCTURALLY VALID TARGET CANDIDATES
#
# Not yet the final "clean" definition.
# Alarm is intentionally NOT excluded here.
# ============================================================

w[
    "Structurally_Valid_Target"
] = (
    w["System_Eligible"]
    &
    ~w["Has_Negative_Interval"]
    &
    ~w["Is_Incomplete"]
    &
    w["Positive_Daily_Volume"]
    &
    w["Daily_Total_Match"]
)

valid_target = w[
    w["Structurally_Valid_Target"]
].copy()

print("\n========================================")
print("STRUCTURALLY VALID TARGET CANDIDATES")
print("========================================")

print(
    f"Valid detector-days: "
    f"{len(valid_target):,}"
)

print(
    f"Percent of system-eligible detector-days: "
    f"{len(valid_target) / eligible_mask.sum() * 100:.2f}%"
)

with_alarm_valid = (
    valid_target["Has_Alarm"]
).sum()

without_alarm_valid = (
    ~valid_target["Has_Alarm"]
).sum()

print(
    f"Valid rows WITH alarm flag: "
    f"{with_alarm_valid:,}"
)

print(
    f"Valid rows WITHOUT alarm flag: "
    f"{without_alarm_valid:,}"
)


# ============================================================
# 9. 15-MINUTE SPARSITY AMONG VALID TARGETS
# ============================================================

valid_target[
    "Zero_Interval_%"
] = (
    valid_target[
        "Zero_Interval_Count"
    ] / 96 * 100
)

print("\n========================================")
print("ZERO-INTERVAL SPARSITY BY VOLUME GROUP")
print("VALID TARGET CANDIDATES ONLY")
print("========================================")

sparsity_by_volume = (
    valid_target
    .groupby(
        "Volume_Group",
        observed=True
    )
    .agg(
        Detector_Days=(
            "Detector_ID",
            "size"
        ),
        Mean_Zero_Interval_Pct=(
            "Zero_Interval_%",
            "mean"
        ),
        Median_Zero_Interval_Pct=(
            "Zero_Interval_%",
            "median"
        ),
        P90_Zero_Interval_Pct=(
            "Zero_Interval_%",
            lambda x: x.quantile(0.90)
        ),
        Mean_Daily_Volume=(
            "QT_VOLUME_24HOUR",
            "mean"
        )
    )
)

print(
    sparsity_by_volume
    .round(2)
    .to_string()
)


# ============================================================
# 10. DETECTOR-LEVEL TARGET AVAILABILITY
# ============================================================

detector_target_availability = (
    w.groupby("Detector_ID")
    .agg(
        Total_Days=("Date", "size"),
        System_Eligible_Days=(
            "System_Eligible",
            "sum"
        ),
        Valid_Target_Days=(
            "Structurally_Valid_Target",
            "sum"
        )
    )
)

detector_target_availability[
    "Valid_Target_%_of_Eligible"
] = (
    detector_target_availability[
        "Valid_Target_Days"
    ]
    /
    detector_target_availability[
        "System_Eligible_Days"
    ]
    * 100
)

print("\n========================================")
print("DETECTOR-LEVEL VALID TARGET AVAILABILITY")
print("========================================")

print(
    detector_target_availability[
        "Valid_Target_%_of_Eligible"
    ]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)


# ============================================================
# 11. SAVE AUDIT TABLES
# ============================================================

quality_combo.to_csv(
    OUTPUT_FOLDER /
    "interval_quality_flag_combinations.csv",
    index=False
)

sparsity_by_volume.to_csv(
    OUTPUT_FOLDER /
    "valid_target_sparsity_by_volume.csv"
)

detector_target_availability.to_csv(
    OUTPUT_FOLDER /
    "detector_valid_target_availability.csv"
)

print(
    "\n✅ Interval-level quality audit complete."
)

In [ ]:
# ============================================================
# NOTEBOOK 2 — DETECTOR-LEVEL SPARSITY & FINAL COHORT AUDIT
# ============================================================

import pandas as pd
import numpy as np

print("========================================")
print("DETECTOR-LEVEL SPARSITY AUDIT")
print("========================================")

# ------------------------------------------------------------
# 1. CLEAN / STRUCTURALLY VALID ROWS
# ------------------------------------------------------------

clean_rows = w[
    w["Structurally_Valid_Target"]
].copy()

# ------------------------------------------------------------
# 2. DETECTOR-LEVEL CLEAN DATA SUMMARY
# ------------------------------------------------------------

detector_sparsity = (
    clean_rows
    .groupby("Detector_ID")
    .agg(
        Clean_Days=(
            "Date",
            "size"
        ),
        Clean_Zero_Intervals=(
            "Zero_Interval_Count",
            "sum"
        ),
        Clean_Positive_Intervals=(
            "Positive_Interval_Count",
            "sum"
        ),
        Mean_Daily_Volume=(
            "QT_VOLUME_24HOUR",
            "mean"
        ),
        Median_Daily_Volume=(
            "QT_VOLUME_24HOUR",
            "median"
        ),
        Median_Daily_Zero_Intervals=(
            "Zero_Interval_Count",
            "median"
        ),
        P90_Daily_Zero_Intervals=(
            "Zero_Interval_Count",
            lambda x: x.quantile(0.90)
        )
    )
    .reset_index()
)

detector_sparsity[
    "Clean_Intervals"
] = (
    detector_sparsity["Clean_Days"] * 96
)

detector_sparsity[
    "Clean_Zero_Interval_%"
] = (
    detector_sparsity["Clean_Zero_Intervals"]
    / detector_sparsity["Clean_Intervals"]
    * 100
)

detector_sparsity[
    "Clean_Positive_Interval_%"
] = (
    detector_sparsity["Clean_Positive_Intervals"]
    / detector_sparsity["Clean_Intervals"]
    * 100
)


# ------------------------------------------------------------
# 3. ATTACH SAMPLE METADATA
# ------------------------------------------------------------

metadata_cols = [
    "Detector_ID",
    "NB_SCATS_SITE",
    "NB_DETECTOR",
    "NM_REGION",
    "Volume_Group",
    "Alarm_Exposure_Group",
    "Sampling_Weight"
]

detector_sparsity = detector_sparsity.merge(
    modeling_sample_weighted[
        metadata_cols
    ],
    on="Detector_ID",
    how="left",
    validate="one_to_one"
)


# ------------------------------------------------------------
# 4. OVERALL SPARSITY DISTRIBUTION
# ------------------------------------------------------------

print("\n========================================")
print("CLEAN ZERO-INTERVAL %")
print("========================================")

print(
    detector_sparsity[
        "Clean_Zero_Interval_%"
    ].describe(
        percentiles=[
            0.01,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)


# ------------------------------------------------------------
# 5. SPARSITY BY ORIGINAL VOLUME GROUP
# ------------------------------------------------------------

print("\n========================================")
print("DETECTOR SPARSITY BY VOLUME GROUP")
print("========================================")

sparsity_group_summary = (
    detector_sparsity
    .groupby(
        "Volume_Group",
        observed=True
    )
    .agg(
        Detectors=("Detector_ID", "size"),
        Median_Zero_Pct=(
            "Clean_Zero_Interval_%",
            "median"
        ),
        Mean_Zero_Pct=(
            "Clean_Zero_Interval_%",
            "mean"
        ),
        P90_Zero_Pct=(
            "Clean_Zero_Interval_%",
            lambda x: x.quantile(0.90)
        ),
        Median_Volume=(
            "Mean_Daily_Volume",
            "median"
        )
    )
)

print(
    sparsity_group_summary
    .round(2)
    .to_string()
)


# ------------------------------------------------------------
# 6. ZERO-SPARSITY THRESHOLD SCENARIOS
# ------------------------------------------------------------

print("\n========================================")
print("ZERO-SPARSITY THRESHOLD SCENARIOS")
print("========================================")

for threshold in [
    90,
    75,
    60,
    50,
    40,
    30,
    25
]:

    temp = detector_sparsity[
        detector_sparsity[
            "Clean_Zero_Interval_%"
        ] <= threshold
    ]

    print(
        f"\nZero intervals <= {threshold}%"
    )

    print(
        f"  Detectors: "
        f"{len(temp):,}"
    )

    print(
        f"  Sites: "
        f"{temp['NB_SCATS_SITE'].nunique():,}"
    )

    print(
        f"  Regions: "
        f"{temp['NM_REGION'].nunique()}"
    )


# ------------------------------------------------------------
# 7. DAILY-VOLUME THRESHOLD SCENARIOS
# ------------------------------------------------------------

print("\n========================================")
print("DAILY-VOLUME THRESHOLD SCENARIOS")
print("========================================")

for threshold in [
    50,
    100,
    250,
    500
]:

    temp = detector_sparsity[
        detector_sparsity[
            "Mean_Daily_Volume"
        ] >= threshold
    ]

    print(
        f"\nMean daily volume >= {threshold}"
    )

    print(
        f"  Detectors: "
        f"{len(temp):,}"
    )

    print(
        f"  Sites: "
        f"{temp['NB_SCATS_SITE'].nunique():,}"
    )

    print(
        f"  Regions: "
        f"{temp['NM_REGION'].nunique()}"
    )


# ------------------------------------------------------------
# 8. PRIMARY COHORT CANDIDATES
# ------------------------------------------------------------

scenario_definitions = {
    "All 800":
        np.ones(
            len(detector_sparsity),
            dtype=bool
        ),

    "Zero <=75%":
        detector_sparsity[
            "Clean_Zero_Interval_%"
        ] <= 75,

    "Zero <=50%":
        detector_sparsity[
            "Clean_Zero_Interval_%"
        ] <= 50,

    "Volume >=100":
        detector_sparsity[
            "Mean_Daily_Volume"
        ] >= 100,

    "Volume >=250":
        detector_sparsity[
            "Mean_Daily_Volume"
        ] >= 250
}

scenario_rows = []

for name, mask in scenario_definitions.items():

    temp = detector_sparsity[
        mask
    ]

    scenario_rows.append({
        "Scenario": name,
        "Detectors": len(temp),
        "Sites": temp[
            "NB_SCATS_SITE"
        ].nunique(),
        "Regions": temp[
            "NM_REGION"
        ].nunique(),
        "Median_Zero_%": temp[
            "Clean_Zero_Interval_%"
        ].median(),
        "Median_Daily_Volume": temp[
            "Mean_Daily_Volume"
        ].median()
    })

scenario_table = pd.DataFrame(
    scenario_rows
)

print("\n========================================")
print("PRIMARY COHORT COMPARISON")
print("========================================")

print(
    scenario_table
    .round(2)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 9. REAL FAILURE-PATTERN LIBRARY
# ------------------------------------------------------------

failure_rows = w[
    w["System_Eligible"]
    &
    w["Has_Negative_Interval"]
].copy()

print("\n========================================")
print("REAL DETECTOR-FAILURE PATTERNS")
print("========================================")

print(
    f"Failure detector-days: "
    f"{len(failure_rows):,}"
)

print(
    f"Detectors with >=1 failure day: "
    f"{failure_rows['Detector_ID'].nunique():,}"
)

print(
    f"Sites with >=1 failure day: "
    f"{failure_rows['NB_SCATS_SITE'].nunique():,}"
)

print("\nNegative intervals per failure day:")

print(
    failure_rows[
        "Negative_Interval_Count"
    ].describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)

print("\nMost common failure lengths:")

print(
    failure_rows[
        "Negative_Interval_Count"
    ]
    .value_counts()
    .head(20)
    .to_string()
)


# ------------------------------------------------------------
# 10. SAVE
# ------------------------------------------------------------

detector_sparsity.to_csv(
    OUTPUT_FOLDER /
    "detector_level_clean_sparsity.csv",
    index=False
)

scenario_table.to_csv(
    OUTPUT_FOLDER /
    "primary_modeling_cohort_scenarios.csv",
    index=False
)

failure_rows[
    [
        "Date",
        "Detector_ID",
        "NB_SCATS_SITE",
        "NB_DETECTOR",
        "NM_REGION",
        "Negative_Interval_Count",
        "CT_ALARM_24HOUR",
        "Is_Incomplete",
        "Volume_Group",
        "Alarm_Exposure_Group"
    ]
].to_csv(
    OUTPUT_FOLDER /
    "real_detector_failure_pattern_library.csv",
    index=False
)

print(
    "\n✅ Detector-level sparsity and failure-pattern audit complete."
)

In [ ]:
# ============================================================
# NOTEBOOK 2 — FREEZE MODELING COHORTS
# + CLASSIFY DETECTOR-DAYS
# + BUILD REAL FAILURE-RUN LIBRARY
# ============================================================

import pandas as pd
import numpy as np

V_COLS = [f"V{i:02d}" for i in range(96)]

print("========================================")
print("FINAL COHORTS & FAILURE PATTERN LIBRARY")
print("========================================")


# ============================================================
# 1. FREEZE PRIMARY COHORT
# ============================================================

primary_cohort = (
    detector_sparsity
    .copy()
    .sort_values("Detector_ID")
    .reset_index(drop=True)
)

primary_cohort["Primary_Cohort"] = True


# ============================================================
# 2. SENSITIVITY COHORT
# At least 50% of clean 15-min intervals are non-zero
# ============================================================

sensitivity_cohort = (
    primary_cohort[
        primary_cohort[
            "Clean_Zero_Interval_%"
        ] <= 50
    ]
    .copy()
    .reset_index(drop=True)
)

sensitivity_ids = set(
    sensitivity_cohort["Detector_ID"]
)

primary_cohort[
    "Sensitivity_Cohort"
] = (
    primary_cohort["Detector_ID"]
    .isin(sensitivity_ids)
)


print("\n========================================")
print("FROZEN MODELING COHORTS")
print("========================================")

print(
    f"Primary cohort: "
    f"{len(primary_cohort):,} detectors"
)

print(
    f"Primary sites: "
    f"{primary_cohort['NB_SCATS_SITE'].nunique():,}"
)

print(
    f"Primary regions: "
    f"{primary_cohort['NM_REGION'].nunique()}"
)

print(
    f"Sensitivity cohort: "
    f"{len(sensitivity_cohort):,} detectors"
)

print(
    f"Sensitivity sites: "
    f"{sensitivity_cohort['NB_SCATS_SITE'].nunique():,}"
)

print(
    f"Sensitivity regions: "
    f"{sensitivity_cohort['NM_REGION'].nunique()}"
)


# ============================================================
# 3. CLASSIFY EVERY SYSTEM-ELIGIBLE DETECTOR-DAY
# ============================================================

eligible = w[
    w["System_Eligible"]
].copy()


def classify_detector_day(row):

    # Real local detector/data failure
    if row["Has_Negative_Interval"]:
        return "Local_Failure"

    # Clean positive traffic target
    if row["Structurally_Valid_Target"]:
        return "Clean_Positive_Target"

    # Complete, internally consistent all-zero/zero-volume day
    if (
        (not row["Has_Negative_Interval"])
        and (not row["Is_Incomplete"])
        and row["Daily_Total_Match"]
        and row["QT_VOLUME_24HOUR"] == 0
    ):
        return "Zero_Volume_Day"

    return "Other_NonClean"


eligible[
    "Detector_Day_Class"
] = eligible.apply(
    classify_detector_day,
    axis=1
)


class_counts = (
    eligible[
        "Detector_Day_Class"
    ]
    .value_counts()
)

print("\n========================================")
print("SYSTEM-ELIGIBLE DETECTOR-DAY CLASSES")
print("========================================")

for name, count in class_counts.items():

    print(
        f"{name:<24}: "
        f"{count:>7,} "
        f"({count / len(eligible) * 100:6.3f}%)"
    )


# ============================================================
# 4. INSPECT OTHER NON-CLEAN ROWS
# ============================================================

other_nonclean = eligible[
    eligible[
        "Detector_Day_Class"
    ] == "Other_NonClean"
].copy()

print("\n========================================")
print("OTHER NON-CLEAN ROWS")
print("========================================")

print(
    f"Rows: "
    f"{len(other_nonclean):,}"
)

if len(other_nonclean) > 0:

    print(
        "\nDaily volume status:"
    )

    print(
        pd.Series({
            "Positive volume": (
                other_nonclean[
                    "QT_VOLUME_24HOUR"
                ] > 0
            ).sum(),

            "Zero volume": (
                other_nonclean[
                    "QT_VOLUME_24HOUR"
                ] == 0
            ).sum(),

            "Negative volume": (
                other_nonclean[
                    "QT_VOLUME_24HOUR"
                ] < 0
            ).sum()
        }).to_string()
    )

    print(
        "\nDaily-total matches:"
    )

    print(
        other_nonclean[
            "Daily_Total_Match"
        ]
        .value_counts()
        .to_string()
    )

    print(
        "\nIncomplete:"
    )

    print(
        other_nonclean[
            "Is_Incomplete"
        ]
        .value_counts()
        .to_string()
    )


# ============================================================
# 5. BUILD CONTIGUOUS FAILURE-RUN LIBRARY
# ============================================================

failure_rows = eligible[
    eligible[
        "Detector_Day_Class"
    ] == "Local_Failure"
].copy()

failure_run_records = []

for row_idx, row in failure_rows.iterrows():

    values = row[V_COLS].to_numpy(
        dtype=np.int32
    )

    failed = values < 0

    failed_indices = np.flatnonzero(
        failed
    )

    if len(failed_indices) == 0:
        continue

    # Split into contiguous runs
    breaks = np.where(
        np.diff(failed_indices) > 1
    )[0]

    run_starts = np.r_[
        0,
        breaks + 1
    ]

    run_ends = np.r_[
        breaks,
        len(failed_indices) - 1
    ]

    runs = []

    for run_number, (s, e) in enumerate(
        zip(run_starts, run_ends),
        start=1
    ):

        start_idx = int(
            failed_indices[s]
        )

        end_idx = int(
            failed_indices[e]
        )

        length = (
            end_idx - start_idx + 1
        )

        runs.append(
            (
                run_number,
                start_idx,
                end_idx,
                length
            )
        )

    number_of_runs = len(runs)

    for (
        run_number,
        start_idx,
        end_idx,
        length
    ) in runs:

        failure_run_records.append({
            "Date": row["Date"],
            "Detector_ID": row["Detector_ID"],
            "NB_SCATS_SITE": row["NB_SCATS_SITE"],
            "NB_DETECTOR": row["NB_DETECTOR"],
            "NM_REGION": row["NM_REGION"],

            "Failure_Run_Number":
                run_number,

            "Runs_On_This_Day":
                number_of_runs,

            "Start_Interval_Index":
                start_idx,

            "End_Interval_Index":
                end_idx,

            "Start_Interval":
                f"V{start_idx:02d}",

            "End_Interval":
                f"V{end_idx:02d}",

            "Run_Length":
                length,

            "Total_Negative_Intervals_On_Day":
                row["Negative_Interval_Count"],

            "CT_ALARM_24HOUR":
                row["CT_ALARM_24HOUR"],

            "Volume_Group":
                row["Volume_Group"],

            "Alarm_Exposure_Group":
                row["Alarm_Exposure_Group"]
        })


failure_runs = pd.DataFrame(
    failure_run_records
)


# ============================================================
# 6. FAILURE-RUN VALIDATION
# ============================================================

print("\n========================================")
print("REAL CONTIGUOUS FAILURE RUNS")
print("========================================")

print(
    f"Failure detector-days: "
    f"{len(failure_rows):,}"
)

print(
    f"Total contiguous failure runs: "
    f"{len(failure_runs):,}"
)

print(
    f"Detectors represented: "
    f"{failure_runs['Detector_ID'].nunique():,}"
)

print(
    f"Regions represented: "
    f"{failure_runs['NM_REGION'].nunique()}"
)


# ============================================================
# 7. RUN LENGTH DISTRIBUTION
# ============================================================

print("\n========================================")
print("FAILURE RUN-LENGTH DISTRIBUTION")
print("========================================")

print(
    failure_runs[
        "Run_Length"
    ].describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)


print("\nMost common contiguous run lengths:")

print(
    failure_runs[
        "Run_Length"
    ]
    .value_counts()
    .head(25)
    .to_string()
)


# ============================================================
# 8. NUMBER OF FAILURE RUNS PER DETECTOR-DAY
# ============================================================

runs_per_day = (
    failure_runs[
        [
            "Date",
            "Detector_ID",
            "Runs_On_This_Day"
        ]
    ]
    .drop_duplicates(
        subset=[
            "Date",
            "Detector_ID"
        ]
    )
)

print("\n========================================")
print("FAILURE RUNS PER DETECTOR-DAY")
print("========================================")

print(
    runs_per_day[
        "Runs_On_This_Day"
    ].value_counts()
    .sort_index()
    .to_string()
)


# ============================================================
# 9. START-POSITION DISTRIBUTION
# ============================================================

print("\n========================================")
print("MOST COMMON FAILURE START INTERVALS")
print("========================================")

print(
    failure_runs[
        "Start_Interval"
    ]
    .value_counts()
    .head(20)
    .to_string()
)


# ============================================================
# 10. CONSISTENCY CHECK
# Sum of all run lengths must equal total number of -1
# intervals in system-eligible failure rows.
# ============================================================

run_length_sum = int(
    failure_runs[
        "Run_Length"
    ].sum()
)

negative_interval_total = int(
    failure_rows[
        "Negative_Interval_Count"
    ].sum()
)

print("\n========================================")
print("FAILURE LIBRARY VALIDATION")
print("========================================")

print(
    f"Sum of contiguous run lengths: "
    f"{run_length_sum:,}"
)

print(
    f"Negative intervals in failure rows: "
    f"{negative_interval_total:,}"
)

print(
    f"Exact match: "
    f"{run_length_sum == negative_interval_total}"
)


# ============================================================
# 11. SAVE FINAL COHORTS
# ============================================================

primary_cohort.to_csv(
    OUTPUT_FOLDER /
    "FINAL_PRIMARY_COHORT_800.csv",
    index=False
)

sensitivity_cohort.to_csv(
    OUTPUT_FOLDER /
    "FINAL_SENSITIVITY_COHORT_ZERO_LE_50.csv",
    index=False
)

eligible[
    [
        "Date",
        "Detector_ID",
        "NB_SCATS_SITE",
        "NB_DETECTOR",
        "NM_REGION",
        "Detector_Day_Class",
        "QT_VOLUME_24HOUR",
        "CT_RECORDS",
        "CT_ALARM_24HOUR",
        "Negative_Interval_Count",
        "Zero_Interval_Count",
        "Positive_Interval_Count",
        "Daily_Total_Match"
    ]
].to_parquet(
    OUTPUT_FOLDER /
    "detector_day_quality_classes.parquet",
    index=False
)

failure_runs.to_csv(
    OUTPUT_FOLDER /
    "REAL_FAILURE_CONTIGUOUS_RUN_LIBRARY.csv",
    index=False
)

print("\n========================================")
print("SAVED")
print("========================================")

print(
    "✅ Primary cohort frozen: 800 detectors"
)

print(
    "✅ Sensitivity cohort frozen"
)

print(
    "✅ Detector-day classes saved"
)

print(
    "✅ Real contiguous failure-pattern library saved"
)

In [ ]:
# ============================================================
# NOTEBOOK 2 — BUILD CLEAN 15-MINUTE TIME SERIES
# MEMORY-SAFE MONTH-BY-MONTH EXPORT
# ============================================================

import pandas as pd
import numpy as np
import gc
from pathlib import Path

V_COLS = [f"V{i:02d}" for i in range(96)]

# ============================================================
# 1. MAKE SURE FINAL QUALITY FIELDS EXIST
# ============================================================

if "Daily_Total_Match" not in w.columns:

    interval_sum = (
        w[V_COLS]
        .to_numpy(dtype=np.int64)
        .sum(axis=1)
    )

    w["Daily_Total_Match"] = (
        interval_sum
        == w["QT_VOLUME_24HOUR"].to_numpy()
    )


if "Structurally_Valid_Target" not in w.columns:

    w["Structurally_Valid_Target"] = (
        w["System_Eligible"]
        &
        ~w["Has_Negative_Interval"]
        &
        ~w["Is_Incomplete"]
        &
        w["Positive_Daily_Volume"]
        &
        w["Daily_Total_Match"]
    )


# ============================================================
# 2. FINAL DETECTOR-DAY CLASSIFICATION
# ============================================================

conditions = [

    ~w["System_Eligible"],

    (
        w["System_Eligible"]
        &
        w["Has_Negative_Interval"]
    ),

    w["Structurally_Valid_Target"],

    (
        w["System_Eligible"]
        &
        ~w["Has_Negative_Interval"]
        &
        ~w["Is_Incomplete"]
        &
        w["Daily_Total_Match"]
        &
        (w["QT_VOLUME_24HOUR"] == 0)
    )
]

choices = [
    "System_Disruption",
    "Local_Failure",
    "Clean_Positive_Target",
    "Zero_Volume_Day"
]

w["Detector_Day_Class"] = np.select(
    conditions,
    choices,
    default="Other_NonClean"
)


# ============================================================
# 3. CREATE NUMERIC DETECTOR CODE
#
# More memory-efficient than repeating long Detector_ID
# strings millions of times.
# ============================================================

detector_mapping = (
    primary_cohort[
        [
            "Detector_ID",
            "NB_SCATS_SITE",
            "NB_DETECTOR",
            "NM_REGION",
            "Volume_Group",
            "Alarm_Exposure_Group",
            "Sampling_Weight",
            "Sensitivity_Cohort"
        ]
    ]
    .sort_values("Detector_ID")
    .reset_index(drop=True)
)

detector_mapping[
    "Detector_Code"
] = np.arange(
    len(detector_mapping),
    dtype=np.int16
)


# Attach code to wide dataset
w = w.merge(
    detector_mapping[
        [
            "Detector_ID",
            "Detector_Code",
            "Sensitivity_Cohort"
        ]
    ],
    on="Detector_ID",
    how="left",
    validate="many_to_one",
    suffixes=("", "_Final")
)

# Handle case where sensitivity field already existed
if "Sensitivity_Cohort_Final" in w.columns:
    w["Sensitivity_Cohort"] = (
        w["Sensitivity_Cohort_Final"]
    )
    w.drop(
        columns=["Sensitivity_Cohort_Final"],
        inplace=True
    )


# ============================================================
# 4. SAVE FINAL ENRICHED WIDE DATASET
# ============================================================

FINAL_WIDE_ENRICHED = (
    OUTPUT_FOLDER /
    "MODELING_WIDE_800_FINAL_ENRICHED.parquet"
)

w.to_parquet(
    FINAL_WIDE_ENRICHED,
    index=False
)


# ============================================================
# 5. VALID OBSERVED GROUND-TRUTH DAYS
# ============================================================

valid_day_classes = [
    "Clean_Positive_Target",
    "Zero_Volume_Day"
]

valid_days = w[
    w["Detector_Day_Class"]
    .isin(valid_day_classes)
].copy()


print("========================================")
print("VALID GROUND-TRUTH DAYS")
print("========================================")

print(
    f"Clean positive detector-days: "
    f"{(valid_days['Detector_Day_Class'] == 'Clean_Positive_Target').sum():,}"
)

print(
    f"Valid zero-volume detector-days: "
    f"{(valid_days['Detector_Day_Class'] == 'Zero_Volume_Day').sum():,}"
)

print(
    f"Total valid detector-days: "
    f"{len(valid_days):,}"
)

EXPECTED_LONG_ROWS = (
    len(valid_days) * 96
)

print(
    f"Expected 15-min observations: "
    f"{EXPECTED_LONG_ROWS:,}"
)


# ============================================================
# 6. OUTPUT FOLDER
# ============================================================

LONG_FOLDER = (
    OUTPUT_FOLDER /
    "Clean_15min_Monthly"
)

LONG_FOLDER.mkdir(
    exist_ok=True
)


# ============================================================
# 7. MONTH-BY-MONTH CONVERSION
# ============================================================

monthly_manifest = []

total_long_rows = 0
total_negative_long = 0

print("\n========================================")
print("BUILDING MONTHLY 15-MINUTE FILES")
print("========================================\n")


months = sorted(
    valid_days["Date"]
    .dt.to_period("M")
    .unique()
)


for period in months:

    month_start = period.start_time

    month_rows = valid_days[
        valid_days["Date"]
        .dt.to_period("M")
        == period
    ].copy()

    month_rows = (
        month_rows
        .sort_values(
            [
                "Date",
                "Detector_Code"
            ]
        )
        .reset_index(drop=True)
    )

    n_days = len(month_rows)

    # --------------------------------------------------------
    # NUMPY-BASED LONG CONVERSION
    # Much lighter than pandas.melt()
    # --------------------------------------------------------

    volumes = (
        month_rows[V_COLS]
        .to_numpy(dtype=np.int16)
        .reshape(-1)
    )

    detector_codes = np.repeat(
        month_rows[
            "Detector_Code"
        ].to_numpy(dtype=np.int16),
        96
    )

    dates = np.repeat(
        month_rows[
            "Date"
        ].to_numpy(
            dtype="datetime64[ns]"
        ),
        96
    )

    interval_index = np.tile(
        np.arange(
            96,
            dtype=np.uint8
        ),
        n_days
    )

    zero_day_flag = np.repeat(
        (
            month_rows[
                "Detector_Day_Class"
            ]
            == "Zero_Volume_Day"
        ).to_numpy(),
        96
    )

    sensitivity_flag = np.repeat(
        month_rows[
            "Sensitivity_Cohort"
        ].to_numpy(dtype=bool),
        96
    )

    # --------------------------------------------------------
    # TIMESTAMP
    #
    # V00 = 00:00–00:15
    # V01 = 00:15–00:30
    # ...
    # V95 = 23:45–24:00
    # --------------------------------------------------------

    timestamps = (
        dates
        +
        interval_index.astype(
            "timedelta64[m]"
        ) * 15
    )

    # --------------------------------------------------------
    # CREATE LEAN LONG TABLE
    # --------------------------------------------------------

    long_month = pd.DataFrame({
        "Timestamp": timestamps,
        "Detector_Code": detector_codes,
        "Interval_Index": interval_index,
        "Volume": volumes,
        "Is_Zero_Volume_Day": zero_day_flag,
        "Sensitivity_Cohort": sensitivity_flag
    })

    # --------------------------------------------------------
    # VALIDATE
    # --------------------------------------------------------

    expected_month_rows = (
        n_days * 96
    )

    negative_count = int(
        (
            long_month["Volume"] < 0
        ).sum()
    )

    total_long_rows += len(
        long_month
    )

    total_negative_long += (
        negative_count
    )

    min_timestamp = (
        long_month[
            "Timestamp"
        ].min()
    )

    max_timestamp = (
        long_month[
            "Timestamp"
        ].max()
    )

    # --------------------------------------------------------
    # SAVE PARQUET
    # --------------------------------------------------------

    month_label = str(period)

    output_file = (
        LONG_FOLDER /
        f"CLEAN_15MIN_{month_label}.parquet"
    )

    long_month.to_parquet(
        output_file,
        index=False
    )

    monthly_manifest.append({
        "Month": month_label,
        "Valid_Detector_Days": n_days,
        "Long_Rows": len(long_month),
        "Expected_Rows": expected_month_rows,
        "Negative_Values": negative_count,
        "Min_Timestamp": min_timestamp,
        "Max_Timestamp": max_timestamp,
        "File": str(output_file)
    })

    print(
        f"{month_label}: "
        f"{n_days:,} detector-days → "
        f"{len(long_month):,} intervals "
        f"| negative={negative_count}"
    )

    del month_rows
    del volumes
    del detector_codes
    del dates
    del interval_index
    del zero_day_flag
    del sensitivity_flag
    del timestamps
    del long_month

    gc.collect()


# ============================================================
# 8. MANIFEST
# ============================================================

manifest = pd.DataFrame(
    monthly_manifest
)

manifest.to_csv(
    OUTPUT_FOLDER /
    "CLEAN_15MIN_monthly_manifest.csv",
    index=False
)


# ============================================================
# 9. HARD VALIDATION
# ============================================================

print("\n========================================")
print("CLEAN 15-MINUTE DATASET VALIDATION")
print("========================================")

print(
    f"Monthly files created: "
    f"{len(manifest)}"
)

print(
    f"Total 15-min rows created: "
    f"{total_long_rows:,}"
)

print(
    f"Expected 15-min rows: "
    f"{EXPECTED_LONG_ROWS:,}"
)

print(
    f"Exact row-count match: "
    f"{total_long_rows == EXPECTED_LONG_ROWS}"
)

print(
    f"Negative values in clean dataset: "
    f"{total_negative_long:,}"
)

print(
    f"Primary detectors represented: "
    f"{valid_days['Detector_Code'].nunique():,}"
)

print(
    f"Sensitivity detectors represented: "
    f"{valid_days.loc[
        valid_days['Sensitivity_Cohort'],
        'Detector_Code'
    ].nunique():,}"
)


# ============================================================
# 10. SAVE DETECTOR CODE MAPPING
# ============================================================

detector_mapping.to_csv(
    OUTPUT_FOLDER /
    "DETECTOR_CODE_MAPPING_800.csv",
    index=False
)


# ============================================================
# 11. FINAL SUMMARY
# ============================================================

print("\n========================================")
print("MONTHLY MANIFEST")
print("========================================")

print(
    manifest[
        [
            "Month",
            "Valid_Detector_Days",
            "Long_Rows",
            "Negative_Values"
        ]
    ].to_string(index=False)
)

print("\n========================================")
print("SAVED")
print("========================================")

print(
    f"Clean monthly 15-min files:\n"
    f"{LONG_FOLDER}"
)

print(
    "\n✅ Clean 15-minute time series created successfully."
)

In [ ]:
# ============================================================
# NOTEBOOK 2 — FINAL CHRONOLOGICAL TRAIN / VAL / TEST SPLIT
# ============================================================

import pandas as pd
from pathlib import Path

print("========================================")
print("FINAL CHRONOLOGICAL DATA SPLIT")
print("========================================")

# ============================================================
# 1. SPLIT DEFINITION
# ============================================================

SPLIT_CONFIG = {
    "Train": {
        "start": pd.Timestamp("2026-01-01"),
        "end":   pd.Timestamp("2026-06-30 23:59:59")
    },
    "Validation": {
        "start": pd.Timestamp("2026-07-01"),
        "end":   pd.Timestamp("2026-07-31 23:59:59")
    },
    "Test": {
        "start": pd.Timestamp("2026-08-01"),
        "end":   pd.Timestamp("2026-08-30 23:59:59")
    }
}

for split, info in SPLIT_CONFIG.items():
    print(
        f"{split:<10}: "
        f"{info['start'].date()} → {info['end'].date()}"
    )


# ============================================================
# 2. MONTHLY PARQUET FILES
# ============================================================

monthly_files = sorted(
    LONG_FOLDER.glob("CLEAN_15MIN_*.parquet")
)

print("\nMonthly files found:", len(monthly_files))


# ============================================================
# 3. FILE → SPLIT MAPPING
# ============================================================

def get_month_from_filename(path):
    # CLEAN_15MIN_2026-01.parquet
    month_text = (
        path.stem
        .replace("CLEAN_15MIN_", "")
    )

    return pd.Period(
        month_text,
        freq="M"
    )


split_records = []

for file in monthly_files:

    period = get_month_from_filename(file)

    month_start = period.start_time
    month_end = period.end_time

    if month_start.month <= 6:
        split = "Train"
    elif month_start.month == 7:
        split = "Validation"
    elif month_start.month == 8:
        split = "Test"
    else:
        split = "Unassigned"

    # Read only lightweight columns
    temp = pd.read_parquet(
        file,
        columns=[
            "Timestamp",
            "Detector_Code",
            "Sensitivity_Cohort"
        ]
    )

    split_records.append({
        "Month": str(period),
        "Split": split,
        "Rows": len(temp),
        "Unique_Detectors":
            temp["Detector_Code"].nunique(),
        "Sensitivity_Detectors":
            temp.loc[
                temp["Sensitivity_Cohort"],
                "Detector_Code"
            ].nunique(),
        "Min_Timestamp":
            temp["Timestamp"].min(),
        "Max_Timestamp":
            temp["Timestamp"].max(),
        "File":
            str(file)
    })

    del temp


file_split_manifest = pd.DataFrame(
    split_records
)


# ============================================================
# 4. MONTHLY MANIFEST
# ============================================================

print("\n========================================")
print("MONTH → SPLIT ASSIGNMENT")
print("========================================")

print(
    file_split_manifest[
        [
            "Month",
            "Split",
            "Rows",
            "Unique_Detectors",
            "Sensitivity_Detectors"
        ]
    ].to_string(index=False)
)


# ============================================================
# 5. SPLIT-LEVEL SUMMARY
# ============================================================

split_summary = (
    file_split_manifest
    .groupby(
        "Split",
        sort=False
    )
    .agg(
        Months=("Month", "count"),
        Rows=("Rows", "sum"),
        Min_Timestamp=(
            "Min_Timestamp",
            "min"
        ),
        Max_Timestamp=(
            "Max_Timestamp",
            "max"
        )
    )
    .reset_index()
)

TOTAL_ROWS = (
    file_split_manifest["Rows"].sum()
)

split_summary["Percent_of_Data"] = (
    split_summary["Rows"]
    / TOTAL_ROWS
    * 100
)

print("\n========================================")
print("SPLIT SUMMARY")
print("========================================")

print(
    split_summary
    .round({
        "Percent_of_Data": 2
    })
    .to_string(index=False)
)


# ============================================================
# 6. TEMPORAL LEAKAGE VALIDATION
# ============================================================

train_max = split_summary.loc[
    split_summary["Split"] == "Train",
    "Max_Timestamp"
].iloc[0]

val_min = split_summary.loc[
    split_summary["Split"] == "Validation",
    "Min_Timestamp"
].iloc[0]

val_max = split_summary.loc[
    split_summary["Split"] == "Validation",
    "Max_Timestamp"
].iloc[0]

test_min = split_summary.loc[
    split_summary["Split"] == "Test",
    "Min_Timestamp"
].iloc[0]


train_val_ok = (
    train_max < val_min
)

val_test_ok = (
    val_max < test_min
)

print("\n========================================")
print("TEMPORAL LEAKAGE CHECK")
print("========================================")

print(
    f"Train ends before Validation begins: "
    f"{train_val_ok}"
)

print(
    f"Validation ends before Test begins: "
    f"{val_test_ok}"
)

print(
    f"Overall chronological split valid: "
    f"{train_val_ok and val_test_ok}"
)


# ============================================================
# 7. DETECTOR COVERAGE BY SPLIT
# ============================================================

print("\n========================================")
print("DETECTOR COVERAGE BY SPLIT")
print("========================================")

for split_name in [
    "Train",
    "Validation",
    "Test"
]:

    files = file_split_manifest.loc[
        file_split_manifest["Split"]
        == split_name,
        "File"
    ]

    detector_set = set()
    sensitivity_set = set()

    for file in files:

        temp = pd.read_parquet(
            file,
            columns=[
                "Detector_Code",
                "Sensitivity_Cohort"
            ]
        )

        detector_set.update(
            temp[
                "Detector_Code"
            ].unique()
        )

        sensitivity_set.update(
            temp.loc[
                temp[
                    "Sensitivity_Cohort"
                ],
                "Detector_Code"
            ].unique()
        )

        del temp

    print(
        f"{split_name:<10}: "
        f"{len(detector_set):,} primary detectors | "
        f"{len(sensitivity_set):,} sensitivity detectors"
    )


# ============================================================
# 8. SAVE SPLIT MANIFESTS
# ============================================================

file_split_manifest.to_csv(
    OUTPUT_FOLDER /
    "FINAL_DATA_SPLIT_FILE_MANIFEST.csv",
    index=False
)

split_summary.to_csv(
    OUTPUT_FOLDER /
    "FINAL_DATA_SPLIT_SUMMARY.csv",
    index=False
)

split_config_table = pd.DataFrame([
    {
        "Split": split,
        "Start_Date": info["start"],
        "End_Date": info["end"]
    }
    for split, info
    in SPLIT_CONFIG.items()
])

split_config_table.to_csv(
    OUTPUT_FOLDER /
    "FINAL_DATA_SPLIT_CONFIGURATION.csv",
    index=False
)


# ============================================================
# 9. FINAL NOTE FOR NOTEBOOK 3
# ============================================================

print("\n========================================")
print("IMPORTANT RULE FOR NOTEBOOK 3")
print("========================================")

print(
    "When creating forecasting sequences:"
)

print(
    "DO NOT allow a sequence to cross a missing timestamp, "
    "failure period, system-disruption gap, or split boundary."
)

print(
    "Every input/output sequence must be chronologically "
    "continuous at exactly 15-minute intervals."
)


print("\n========================================")
print("NOTEBOOK 2 STATUS")
print("========================================")

print(
    "✅ Detector selection complete"
)

print(
    "✅ Primary cohort = 800 detectors"
)

print(
    "✅ Sensitivity cohort = 738 detectors"
)

print(
    "✅ Clean 15-minute dataset complete"
)

print(
    "✅ Real failure-pattern library complete"
)

print(
    "✅ Chronological Train / Validation / Test split frozen"
)